# 🔬 30 · Single-cell RNA-seq: from counts to cell types
### Quality control, doublets, normalisation, feature selection, PCA, graphs, clustering, marker genes, annotation and condition comparisons — the standard pipeline built from scratch and graded against a known truth

**BiologyToML** · Level 2 · Research frontier · Part G — Cells and tissues · Module 30 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/30_single_cell_rnaseq.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, pandas, Matplotlib, scikit-learn (k-NN, k-means, GMM, t-SNE, ARI), NetworkX (Louvain) |
| **Prerequisites** | Level 1 (especially Modules 08, 12–14 and 23) and [Module 26](26_glm_count_data.ipynb) (negative-binomial GLMs, pseudobulk); comfortable with NumPy and pandas |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **Builds on** | [08 · k-NN](08_knn_naive_bayes.ipynb), [12 · Clustering](12_clustering_kmeans_hierarchical.ipynb), [13 · Mixture models](13_mixture_models_em.ipynb), [14 · PCA](14_pca_dimensionality_reduction.ipynb), [23 · Model selection](23_model_selection.ipynb), [26 · GLMs for counts](26_glm_count_data.ipynb) |
| **Research frontier covered** | single-cell atlases (Human Cell Atlas), quality control, normalisation and feature selection, graph-based clustering, cell-type annotation, condition comparisons — see [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **All data are SIMULATED** with the toolkit's `simulate_scrna`: 4,000 PBMC-like cells from six donors (three control,
> three interferon-stimulated). The hidden truth includes each cell's type, which droplets are doublets or damaged
> cells, which cells are cycling, the marker genes of each type and the true stimulation effect of every gene in every
> type. Gene names follow real immunology so that plots read naturally, but the expression values are simulated. Real
> data add ambient RNA, batch effects, continuous cell states and no answer key — see Section 9.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** how droplet-based scRNA-seq turns cells into a sparse matrix of UMI counts, and which artefacts it contains (empty droplets and ambient RNA, doublets, damaged cells).
2. **Perform** quality control with median-absolute-deviation thresholds and **implement** a Scrublet-style doublet detector; **measure** their precision and recall against the truth.
3. **Normalise** counts (library size + log; analytic Pearson residuals) and **select** highly variable genes in three ways, scoring each by how many true marker genes it recovers.
4. **Reduce** dimensionality with PCA, **build** a k-nearest-neighbour graph and **visualise** it with t-SNE — and **quantify** what a 2-D embedding preserves and what it distorts.
5. **Cluster** cells by modularity optimisation (Louvain), **compare** with k-means and Gaussian mixtures using the adjusted Rand index, and **explain** the resolution parameter.
6. **Find** marker genes with a from-scratch Wilcoxon rank-sum test, **draw** a dot plot, **annotate** clusters and **measure** the annotation accuracy.
7. **Compare** conditions per cell type with pseudobulk negative-binomial GLMs versus cell-level tests, and **test** changes in cell-type composition with donors as replicates.
8. **Connect** every step to Scanpy and Seurat and to atlas-scale resources (CELLxGENE, the Human Cell Atlas).

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit | 5 min |
| 1 · The assay and the matrix | droplets, barcodes, UMIs; sparsity and zeros | 15 min |
| 2 · Quality control | QC metrics, MAD thresholds, a doublet detector; 🧪 precision/recall | 25 min |
| 3 · Normalisation and feature selection | log-normalisation, Pearson residuals, HVGs; 🧪 marker recovery | 25 min |
| 4 · Dimensionality reduction | PCA, k-NN graph, t-SNE; 🧪 what embeddings preserve | 20 min |
| 5 · Clustering | modularity and Louvain vs k-means and GMM; 🧪 ARI | 25 min |
| 6 · Marker genes and annotation | rank-sum tests, dot plot, marker scores; 🧪 accuracy | 20 min |
| 7 · Comparing conditions | pseudobulk vs cell-level DE in every cell type; 🧪 FDR and power | 20 min |
| 8 · Composition | proportions per donor, a planted change, the compositional trap | 10 min |
| 9 · 🧭 From toy to real data | tools, atlases, what goes wrong | 10 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 5 min |

## 🧫 The biological question

A drop of blood contains a whole immune system: **T cells** (CD4 "helpers" that coordinate responses and CD8 "killers"
that destroy infected cells), **B cells** that make antibodies, **natural killer (NK) cells**, **monocytes** that patrol
and eat, and rare **dendritic cells** that present fragments of pathogens to T cells. Bulk RNA-seq of peripheral blood
mononuclear cells (PBMCs) averages all of them; a change in "blood expression" could mean that every cell changed, or
only that the *mix* of cells changed. **Single-cell RNA-seq** measures each cell separately.

Kang *et al.* (2018) sequenced PBMCs from eight donors, with and without stimulation by **interferon-β**, the antiviral
alarm signal. They pooled the donors' cells in the same droplet runs and used each donor's natural genetic variants to
tell which cell came from whom — which also revealed droplets that contained cells of two different people (doublets).
The data set became a standard benchmark: interferon switches on a shared antiviral programme in every cell type, plus
responses specific to particular types, monocytes above all.

Our SIMULATED data mimic that design: six donors (three control, three stimulated), six cell types in roughly
realistic proportions (many CD4 T cells, few dendritic cells), and the usual artefacts. The questions of this notebook
are the questions of every single-cell study:

1. Which droplets contain one healthy cell?
2. Which cell types are present, and how do we recognise them?
3. How does each cell type respond to the stimulus — and did the mix of cell types change?

At each step we will build the method from scratch and grade it against the hidden truth, something real data never
allow.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn"), ("networkx", "networkx")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from scipy import special, stats
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`simulate_scrna` generates the single-cell experiment with its hidden truth, `log_normalize` is the standard log(1 + scaled count) transform, `pca` does PCA by SVD, `plot_embedding` scatters 2-D embeddings by label, `benjamini_hochberg` adjusts p-values, `precision_recall_f1` grades binary calls, and `PBMC_TYPES` lists the six cell types with their canonical marker genes. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MITO_GENES = ["MT-ND1", "MT-ND2", "MT-CO1", "MT-CO2", "MT-ATP8", "MT-ATP6", "MT-CO3", "MT-ND3",
              "MT-ND4L", "MT-ND4", "MT-ND5", "MT-ND6", "MT-CYB"]


INTERFERON_GENES = ["ISG15", "IFI6", "IFIT1", "IFIT3", "MX1", "OAS1", "IFI44L", "RSAD2", "IRF7", "STAT1"]


CYCLING_GENES = ["MKI67", "TOP2A", "CCNB1", "CDK1", "UBE2C", "BIRC5", "TYMS", "PCNA"]


PBMC_TYPES = {
    "CD4 T": ("lymphoid", ["CD3E", "CD3D", "IL7R", "CD4"], 0.5),
    "CD8 T": ("lymphoid", ["CD3E", "CD3D", "CD8A", "CD8B"], 0.5),
    "NK": ("lymphoid", ["NKG7", "GNLY", "KLRD1", "PRF1"], 2.2),
    "B": ("lymphoid", ["MS4A1", "CD79A", "CD79B", "CD19"], 2.4),
    "Monocyte": ("myeloid", ["CD14", "LYZ", "S100A8", "S100A9"], 2.6),
    "Dendritic": ("myeloid", ["FCER1A", "CST3", "CLEC10A", "HLA-DQA1"], 2.2),
}


def sample_negative_binomial(rng: np.random.Generator, mu, theta):
    """Negative-binomial counts with mean ``mu`` and inverse dispersion ``theta``.

    Variance = mu + mu²/theta (large theta → Poisson).  Drawn as a Gamma–Poisson mixture:
    each count is Poisson with a rate that itself varies from cell to cell.
    """
    mu = np.asarray(mu, float)
    theta = np.broadcast_to(np.asarray(theta, float), mu.shape)
    return rng.poisson(rng.gamma(theta, np.where(mu > 0, mu, 0.0) / theta)).astype(np.int32)


def make_expression_programs(rng: np.random.Generator, n_genes: int, cell_types: dict,
                             n_extra_markers: int = 20, extra_names=()):
    """Log-scale expression profiles (one row per cell type) for a simulated tissue.

    ``cell_types`` maps a type name to (lineage, named marker genes, marker strength).  Every
    type gets its named markers plus ``n_extra_markers`` anonymous ones; types of the same
    lineage share 20 lineage genes.  ``extra_names`` are reserved gene names (placed after the
    markers) that the caller gives its own behaviour.  Returns ``(log_profiles, genes, marker_of)``.
    """
    named = []
    for _, markers, _ in cell_types.values():
        named += [g for g in markers if g not in named]
    reserved = named + [g for g in extra_names if g not in named]
    if len(reserved) > n_genes:
        raise ValueError("n_genes is too small for the requested marker genes")
    genes = np.array(reserved + [f"Gene{j:04d}" for j in range(1, n_genes - len(reserved) + 1)], dtype=object)
    free = np.arange(len(reserved), n_genes)                       # anonymous genes available
    base = rng.normal(0.0, 2.0, n_genes)                           # gene-specific baseline (log scale)
    base[:len(named)] -= 2.0                                       # named markers are low elsewhere
    profiles = np.tile(base, (len(cell_types), 1)) + rng.normal(0, 0.15, (len(cell_types), n_genes))
    marker_of = np.array([""] * n_genes, dtype=object)
    lineages = sorted({v[0] for v in cell_types.values()})
    lineage_genes = {lin: rng.choice(free, 20, replace=False) for lin in lineages}
    for t, (name, (lineage, markers, strength)) in enumerate(cell_types.items()):
        profiles[t, lineage_genes[lineage]] += rng.normal(1.2, 0.3, 20)
        idx = [int(np.flatnonzero(genes == g)[0]) for g in markers]
        profiles[t, idx] += strength + 2.0 + rng.normal(0, 0.2, len(idx))
        extra = rng.choice(free, n_extra_markers, replace=False)
        profiles[t, extra] += rng.normal(0.6 + strength / 2, 0.3, n_extra_markers)
        for j in list(idx) + list(extra):
            marker_of[j] = name if marker_of[j] == "" else marker_of[j] + "/" + name
    return profiles, genes, marker_of


def simulate_scrna(rng: np.random.Generator, n_cells: int = 3000, n_genes: int = 1000, cell_types=None,
                   proportions=None, n_donors: int = 6, n_batches: int = 2, confounded: bool = False,
                   depth: float = 1500, batch_effect: float = 0.25, donor_effect: float = 0.15,
                   stim_effect: float = 1.0, theta: float = 8.0, doublet_rate: float = 0.0,
                   damaged_rate: float = 0.0, cycling_rate: float = 0.0, drop_type_in_batch=None) -> dict:
    """Simulate single-cell RNA-seq UMI counts for a PBMC-like sample collection.

    Design: ``n_donors`` donors, the first half *control* and the second half *stimulated*
    (an interferon response in every cell type plus a monocyte-specific response).  Donor
    ``d`` is sequenced in batch ``d % n_batches`` — or, with ``confounded=True``, every control
    donor in batch 1 and every stimulated donor in batch 2 (a design flaw to learn from).
    Counts are negative binomial around ``depth`` UMIs per cell.  Optional artefacts for
    quality control: doublets, damaged cells (high mitochondrial fraction), cycling cells.
    ``drop_type_in_batch=("B", 1)`` removes B cells from batch 1 (index 1, i.e. "batch2").

    Expression probabilities are renormalised within each cell (a fixed-size library, as in
    sequencing), so in stimulated cells the genes with ``true_logfc == 0`` lose a few per cent of
    their share (about −0.04 on the natural-log scale) to the induced genes: a compositional
    effect that is also present in real data.

    Returns a dict with ``counts`` (cells × genes, int32), ``obs`` (per-cell table with the
    true cell type, donor, batch, condition, …), ``var`` (per-gene table), ``true_logfc``
    (cell type × gene stimulation effect, natural log) and ``log_profiles``.
    """
    cell_types = PBMC_TYPES if cell_types is None else cell_types
    names = list(cell_types)
    T = len(names)
    p = np.full(T, 1 / T) if proportions is None else np.asarray(proportions, float) / np.sum(proportions)
    extra = MITO_GENES + INTERFERON_GENES + CYCLING_GENES
    profiles, genes, marker_of = make_expression_programs(rng, n_genes, cell_types, extra_names=extra)
    G = len(genes)
    is_mito, is_ifn, is_cyc = (np.isin(genes, MITO_GENES), np.isin(genes, INTERFERON_GENES),
                               np.isin(genes, CYCLING_GENES))
    profiles[:, is_mito] = rng.normal(3.0, 0.3, is_mito.sum())          # ≈ 4 % of UMIs are mitochondrial
    profiles[:, is_ifn | is_cyc] = rng.normal(-1.0, 0.4, (T, (is_ifn | is_cyc).sum()))
    # stimulation: interferon genes up everywhere + 40 monocyte-specific responders (up or down)
    true_logfc = np.zeros((T, G))
    true_logfc[:, is_ifn] = stim_effect * rng.normal(1.6, 0.3, is_ifn.sum())
    if "Monocyte" in names:
        responders = rng.choice(np.flatnonzero((marker_of == "") & ~is_mito & ~is_ifn & ~is_cyc), 40, replace=False)
        true_logfc[names.index("Monocyte"), responders] = stim_effect * rng.choice([-1, 1], 40) * rng.normal(1.2, 0.3, 40)
    # donors, batches, conditions
    donor = rng.integers(0, n_donors, n_cells)
    stimulated = donor >= n_donors // 2
    batch = stimulated.astype(int) if confounded else donor % n_batches
    batch = np.minimum(batch, n_batches - 1)
    cell_type = rng.choice(T, n_cells, p=p)
    if drop_type_in_batch is not None:
        t_drop, b_drop = names.index(drop_type_in_batch[0]), drop_type_in_batch[1]
        bad = (cell_type == t_drop) & (batch == b_drop)
        cell_type[bad] = rng.choice(np.delete(np.arange(T), t_drop), bad.sum(), p=np.delete(p, t_drop) / (1 - p[t_drop]))
    batch_shift = rng.normal(0, batch_effect, (n_batches, G))
    batch_depth = np.exp(rng.normal(0, 0.3, n_batches))
    donor_shift = rng.normal(0, donor_effect, (n_donors, G))
    log_rate = (profiles[cell_type] + batch_shift[batch] + donor_shift[donor]
                + true_logfc[cell_type] * stimulated[:, None])
    cycling = rng.random(n_cells) < cycling_rate
    log_rate[np.ix_(cycling, np.flatnonzero(is_cyc))] += 3.0
    damaged = rng.random(n_cells) < damaged_rate
    log_rate[np.ix_(damaged, np.flatnonzero(is_mito))] += 2.2
    lib = depth * batch_depth[batch] * np.exp(rng.normal(0, 0.35, n_cells))
    lib[damaged] *= 0.35
    prob = np.exp(log_rate - log_rate.max(axis=1, keepdims=True))
    prob /= prob.sum(axis=1, keepdims=True)
    theta_g = theta * np.exp(rng.normal(0, 0.5, G))
    mu = lib[:, None] * prob
    doublet = rng.random(n_cells) < doublet_rate
    partner = rng.integers(0, n_cells, n_cells)
    mu[doublet] = 0.9 * mu[doublet] + 0.9 * mu[partner[doublet]]       # two cells in one droplet
    counts = sample_negative_binomial(rng, mu, theta_g)
    obs = pd.DataFrame({
        "cell_type": np.where(doublet, "Doublet", np.array(names, dtype=object)[cell_type]),
        "true_type": np.array(names, dtype=object)[cell_type],
        "lineage": np.array([cell_types[n][0] for n in names], dtype=object)[cell_type],
        "donor": [f"donor{d + 1}" for d in donor],
        "condition": np.where(stimulated, "stimulated", "control"),
        "batch": [f"batch{b + 1}" for b in batch],
        "n_counts": counts.sum(axis=1),
        "n_genes": (counts > 0).sum(axis=1),
        "pct_mito": 100 * counts[:, is_mito].sum(axis=1) / np.maximum(counts.sum(axis=1), 1),
        "is_doublet": doublet, "is_damaged": damaged, "is_cycling": cycling,
    }, index=[f"cell{i:05d}" for i in range(n_cells)])
    var = pd.DataFrame({"marker_of": marker_of, "is_mito": is_mito, "is_interferon": is_ifn,
                        "is_cycling": is_cyc, "theta": theta_g}, index=genes)
    return dict(counts=counts, obs=obs, var=var, log_profiles=pd.DataFrame(profiles, index=names, columns=genes),
                true_logfc=pd.DataFrame(true_logfc, index=names, columns=genes))


def confusion_counts(y_true, y_pred) -> dict:
    """True/false positives/negatives for binary labels (1 = positive)."""
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    return dict(TP=int(np.sum((y_pred == 1) & (y_true == 1))), FP=int(np.sum((y_pred == 1) & (y_true == 0))),
                FN=int(np.sum((y_pred == 0) & (y_true == 1))), TN=int(np.sum((y_pred == 0) & (y_true == 0))))


def precision_recall_f1(y_true, y_pred):
    """Precision TP/(TP+FP), recall TP/(TP+FN) and their harmonic mean F1 (0 when undefined)."""
    c = confusion_counts(y_true, y_pred)
    precision = c["TP"] / (c["TP"] + c["FP"]) if c["TP"] + c["FP"] else 0.0
    recall = c["TP"] / (c["TP"] + c["FN"]) if c["TP"] + c["FN"] else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


def log_normalize(counts, target_sum: float = 1e4):
    """Library-size normalisation + log: log(1 + target_sum · count / total count of the cell)."""
    counts = np.asarray(counts, float)
    total = counts.sum(axis=1, keepdims=True)
    return np.log1p(target_sum * counts / np.where(total > 0, total, 1.0))


def pca(X, n_components: int = 2):
    """Principal component analysis by SVD of the column-centred data.

    Returns ``(scores, components, explained_variance_ratio)``: scores are N × k, components k × J.
    """
    X = np.asarray(X, float)
    Xc = X - X.mean(axis=0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    var = S ** 2 / (len(X) - 1)
    return U[:, :n_components] * S[:n_components], Vt[:n_components], var[:n_components] / var.sum()


def benjamini_hochberg(pvalues) -> np.ndarray:
    """Benjamini–Hochberg adjusted p-values (q-values): reject where q ≤ α to control the FDR at α."""
    p = np.asarray(pvalues, float)
    order = np.argsort(p)
    ranked = p[order] * len(p) / np.arange(1, len(p) + 1)
    q = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty_like(p)
    out[order] = np.minimum(q, 1.0)
    return out


CLASS_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


def plot_embedding(Z, labels, ax=None, title: str = "", s: float = 4, legend: bool = True, cmap=None):
    """Scatter a 2-D embedding (PCA, t-SNE, latent space, …) coloured by a categorical or numeric label."""
    Z = np.asarray(Z, float)
    labels = np.asarray(labels)
    ax = ax or plt.gca()
    if labels.dtype.kind in "fc":
        sc = ax.scatter(Z[:, 0], Z[:, 1], c=labels, s=s, cmap=cmap or "viridis", linewidths=0)
        plt.colorbar(sc, ax=ax, shrink=0.8)
    else:
        palette = plt.get_cmap("tab10" if len(np.unique(labels)) <= 10 else "tab20")
        for k, lab in enumerate(np.unique(labels)):
            m = labels == lab
            ax.scatter(Z[m, 0], Z[m, 1], s=s, color=palette(k % palette.N), label=str(lab), linewidths=0)
        if legend:
            ax.legend(markerscale=3, fontsize=7, frameon=False, loc="best")
    ax.set_xticks([]); ax.set_yticks([])
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['sample_negative_binomial', 'make_expression_programs', 'simulate_scrna', 'confusion_counts', 'precision_recall_f1', 'log_normalize', 'pca', 'benjamini_hochberg', 'plot_embedding']))

---
## 1 · The assay and the matrix

> 📍 **Where we are.** [Module 26](26_glm_count_data.ipynb) analysed bulk counts — one number per gene per *sample*. Now
> every *cell* gets its own vector of counts. First we need to know how those counts are made, because each artefact we
> will clean up later is a consequence of the chemistry.

### 🔬 Assay box — droplet single-cell RNA-seq (e.g. 10x Genomics Chromium)

- **How it works.** A suspension of cells flows through a microfluidic chip together with gel **beads** and oil. Most
  droplets end up empty, some get one bead and one cell. Each bead carries millions of DNA oligonucleotides with three
  parts: a **cell barcode** (identical on all oligos of a bead — it labels the droplet), a **unique molecular identifier
  (UMI)** (a random sequence, different on every oligo — it labels each captured mRNA molecule) and a poly-T tail that
  grabs the poly-A tails of mRNAs. The cell is lysed inside its droplet, its mRNAs are tagged, and everything is pooled
  and sequenced. Reads with the same barcode come from the same droplet; reads with the same barcode, UMI and gene come
  from the same original molecule (PCR copies are collapsed), so the counts are **molecule counts**.
- **Typical sizes.** Thousands to tens of thousands of cells per run (millions in atlas projects), ~20,000–30,000
  genes, and only a few thousand UMIs per cell — a small fraction of the cell's mRNA molecules is captured and
  sequenced. Most entries of the matrix are zero.
- **Typical artefacts.** *Empty droplets* with only free-floating (ambient) RNA from broken cells; *doublets*, two
  cells in one droplet (a few per cent, more when more cells are loaded); *damaged cells* whose cytoplasmic mRNA leaked
  out while mitochondrial transcripts, protected inside mitochondria, stayed — hence a high mitochondrial fraction; and
  cell-cycle and stress signatures from sample handling.

The schematic below is drawn with Matplotlib.

In [ ]:
# ── 1.1 What can happen in a droplet (schematic) ──
fig, ax = plt.subplots(figsize=(11, 2.5))
ax.set_xlim(0, 11); ax.set_ylim(0.35, 3.05); ax.axis("off"); ax.set_aspect("equal")
cases = [(1.1, "one cell + one bead\n→ a good cell", ["cell"], True),
         (3.3, "bead, no cell\n→ empty droplet (ambient RNA)", [], True),
         (5.5, "two cells + one bead\n→ DOUBLET", ["cell", "cell"], True),
         (7.7, "broken cell + bead\n→ damaged: high % mito", ["broken"], True),
         (9.9, "cell, no bead\n→ never sequenced", ["cell"], False)]
amb = np.random.default_rng(SEED).uniform(-0.45, 0.45, (12, 2))
for x, label, contents, bead in cases:
    ax.add_patch(plt.Circle((x, 1.9), 0.8, fc="#deebf7", ec="#3182bd", lw=1.5))          # the oil droplet
    for k, c in enumerate(contents):
        cx = x - 0.25 + 0.42 * k - 0.15 * (len(contents) - 1)
        ax.add_patch(plt.Circle((cx, 2.05 - 0.15 * k), 0.27, fc="#a1d99b" if c == "cell" else "white",
                                ec="#31a354", lw=1.5, ls="-" if c == "cell" else "--"))
    if not contents or contents == ["broken"]:                                            # free RNA molecules
        ax.scatter(x + amb[:, 0], 1.9 + amb[:, 1], s=6, color="#31a354")
    if bead:
        ax.add_patch(plt.Circle((x + 0.38, 1.55), 0.17, fc="#fdae6b", ec="#e6550d", lw=1.2))
    ax.text(x, 0.95, label, ha="center", va="top", fontsize=8)
ax.text(5.5, 2.95, "green = cell (dashed: broken) · dots = free RNA · orange = bead carrying barcode + UMI oligos",
        ha="center", fontsize=8)
plt.show()

In [ ]:
# ── 1.2 The SIMULATED experiment: 4,000 droplets from six donors ──
PROPORTIONS = [0.35, 0.17, 0.12, 0.12, 0.21, 0.03]          # CD4 T, CD8 T, NK, B, monocyte, dendritic
scd = simulate_scrna(np.random.default_rng(SEED + 1), n_cells=4000, n_genes=1500, n_batches=1, n_donors=6,
                     proportions=PROPORTIONS, doublet_rate=0.05, damaged_rate=0.05, cycling_rate=0.05)
counts, obs, var = scd["counts"], scd["obs"].copy(), scd["var"]
gene_names = var.index.to_numpy(str)
print(f"SIMULATED UMI matrix: {counts.shape[0]} cells × {counts.shape[1]} genes, {counts.dtype}")
print(f"zeros: {np.mean(counts == 0):.1%} of all entries")
print(f"UMIs per cell: median {np.median(obs.n_counts):.0f} (5th–95th percentile {np.percentile(obs.n_counts, 5):.0f}–"
      f"{np.percentile(obs.n_counts, 95):.0f}); genes detected per cell: median {np.median(obs.n_genes):.0f}")
print("\ncells per donor and condition:\n", pd.crosstab(obs.donor, obs.condition).to_string())
print("\nhidden truth — cell types:", obs.true_type.value_counts().to_dict())
print("hidden truth — artefacts:", {k: int(obs[k].sum()) for k in ["is_doublet", "is_damaged", "is_cycling"]})
top = np.argsort(counts.sum(axis=0))[::-1][:5]
print("\nmost abundant genes (share of all UMIs):", {str(gene_names[j]): round(float(counts[:, j].sum() / counts.sum()), 3) for j in top})

🔍 **What just happened?** We have 4,000 droplets × 1,500 genes, and about seven out of ten entries are zero. A typical
cell has about 2,000 UMIs spread over a few hundred detected genes — out of the tens to hundreds of thousands of mRNA
molecules a real cell contains. Because the counts are spread so thinly, most genes are zero in most cells. The design has about
650 cells per donor, and the hidden truth tells us that roughly 5 % of the droplets are doublets, 5 % are damaged and
5 % of the cells are cycling. A single highly expressed gene takes over a tenth of all UMIs; in real PBMC data the top
spots are usually taken by mitochondrial, ribosomal-protein and a few very abundant transcripts.

### Are the zeros "dropouts"?

Early papers called the many zeros "dropouts" and modelled them with zero-inflated distributions, as if the assay
randomly lost molecules beyond normal counting noise. If a gene has mean $\mu$ UMIs per cell, the probability of a zero
is $e^{-\mu}$ under the Poisson and $(1 + \mu/\theta)^{-\theta}$ under the negative binomial of Module 26. Comparing the
observed fraction of zeros with these curves is the diagnostic Svensson (2020) applied to real droplet data.

In [ ]:
# ── 1.3 Library sizes and zeros ──
mean_g = counts.mean(axis=0)
zero_g = (counts == 0).mean(axis=0)
grid = np.logspace(-3, 2.5, 200)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].hist(np.log10(obs.n_counts), bins=60, color="0.6")
axes[0].set_xlabel("log10 UMIs per droplet"); axes[0].set_ylabel("droplets")
axes[0].set_title("SIMULATED library sizes vary several-fold between droplets")
axes[1].scatter(mean_g, zero_g, s=5, color="0.4", alpha=0.6, label="genes")
axes[1].plot(grid, np.exp(-grid), color=CLASS_COLORS[0], lw=2, label="Poisson: $e^{-\\mu}$")
axes[1].plot(grid, (1 + grid / 8) ** -8, color=CLASS_COLORS[4], lw=2, label="negative binomial, θ = 8")
axes[1].set_xscale("log"); axes[1].set_xlabel("mean UMIs per cell (gene)"); axes[1].set_ylabel("fraction of cells with 0")
axes[1].set_title("zeros are mostly what counting predicts"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()
markers_named = [g for _, m, _ in PBMC_TYPES.values() for g in m]
mk = np.isin(gene_names, markers_named)
excess = zero_g - (1 + mean_g / 8) ** -8
print(f"median excess zeros over the NB curve: all genes {np.median(excess):+.3f}, canonical marker genes {np.median(excess[mk]):+.3f}")

🔍 **What just happened?** Library sizes span about a four-fold range for the central 90 % of droplets (924–3,873 UMIs)
and more than ten-fold overall — in real data partly biology (cell size and type) and partly capture efficiency — so raw
counts are not comparable between cells (Section 3). The fraction of zeros falls
smoothly with the mean, and most genes lie close to the negative-binomial curve; nobody needs a separate "dropout"
process to explain them. The genes with clearly *more* zeros than the curve are the canonical **marker genes**: they are
high in one cell type and nearly absent in the others, so averaged over all cells they have many zeros *for a biological
reason*. That is the conclusion Svensson (2020) reached for real droplet UMI data: zeros mostly reflect low expression
and cell-type heterogeneity, not technical zero inflation. (Our simulator has no zero inflation by construction; on real
data the same plot is a genuine test.)

✏️ **Your turn.** Recompute `mean_g` and `zero_g` using only the true CD4 T cells (`obs.true_type == "CD4 T"`). Do the
marker genes still stand out above the curve? Why not?

---
## 2 · Quality control

> 📍 **Where we are.** Before any biology we must remove droplets that do not contain one intact cell. Two problems
> need two different tools: damaged cells are visible in simple per-cell summaries; doublets need a model.

### Three QC metrics and robust thresholds

For every droplet we compute the **total UMIs**, the **number of detected genes** and the **percentage of UMIs from
mitochondrial genes** (names starting with `MT-`). Damaged cells have few UMIs, few genes and a high mitochondrial
percentage. Fixed thresholds ("discard cells with > 10 % mito") do not transfer between tissues and protocols, so
current best practice (Heumos *et al.* 2023) flags **outliers** relative to the data set, using the **median absolute
deviation (MAD)**:

$$\text{MAD}(x) = 1.4826 \cdot \operatorname{median}_i\,\big|x_i - \operatorname{median}(x)\big|, \qquad \text{flag cell } i \text{ if } x_i < \operatorname{median}(x) - k\cdot\text{MAD}(x) .$$

*In words:* the MAD is a robust standard deviation (the factor 1.4826 makes it equal to the SD for Gaussian data) that is
not inflated by the outliers we are hunting; a cell is flagged if it lies more than $k$ MADs on the bad side of the
median. We use $k = 5$ for (log) counts and genes — lenient, so that small but healthy cell types survive — and $k = 3$
for the mitochondrial percentage.

In [ ]:
# ── 2.1 QC metrics and MAD-based outlier flags ──
def mad_outlier(x, k, side):
    """Flag values more than k robust standard deviations (MADs) below ('low') or above ('high') the median."""
    med, mad = np.median(x), stats.median_abs_deviation(x, scale="normal")
    return x < med - k * mad if side == "low" else x > med + k * mad


log_counts, log_genes, pct_mito = np.log10(obs.n_counts.values), np.log10(obs.n_genes.values), obs.pct_mito.values
flags = pd.DataFrame({"low counts": mad_outlier(log_counts, 5, "low"), "low genes": mad_outlier(log_genes, 5, "low"),
                      "high % mito": mad_outlier(pct_mito, 3, "high")})
damaged_call = flags.any(axis=1).values
print("flagged by each rule:", flags.sum().to_dict(), "→ any:", int(damaged_call.sum()))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
is_dam = obs.is_damaged.values
for ax, x, name, k, side in [(axes[0], log_counts, "log10 UMIs", 5, "low"), (axes[1], log_genes, "log10 genes", 5, "low"),
                             (axes[2], pct_mito, "% mitochondrial UMIs", 3, "high")]:
    bins = np.linspace(x.min(), x.max(), 60)
    ax.hist([x[~is_dam], x[is_dam]], bins=bins, stacked=True, color=["0.65", CLASS_COLORS[1]],
            label=["intact (hidden truth)", "damaged (hidden truth)"])
    med, mad = np.median(x), stats.median_abs_deviation(x, scale="normal")
    ax.axvline(med - k * mad if side == "low" else med + k * mad, color="k", ls="--", lw=1.2)
    ax.set_xlabel(name); ax.set_title(f"threshold (dashed): median {'−' if side == 'low' else '+'} {k} MAD")
axes[0].set_ylabel("droplets"); axes[2].set_yscale("log"); axes[2].set_ylabel("droplets (log scale)"); axes[2].legend(fontsize=7)
plt.suptitle("SIMULATED QC metrics with robust thresholds", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── 2.2 🧪 Ground-truth check for the damaged-cell rule ──
rows = {}
for k_counts, k_mito in [(3, 3), (5, 3), (5, 5)]:
    call = (mad_outlier(log_counts, k_counts, "low") | mad_outlier(log_genes, k_counts, "low")
            | mad_outlier(pct_mito, k_mito, "high"))
    p, r, f = precision_recall_f1(is_dam.astype(int), call.astype(int))
    rows[f"counts/genes {k_counts} MAD, mito {k_mito} MAD"] = {"flagged": int(call.sum()), "precision": p, "recall": r, "F1": f}
print(f"🧪 {int(is_dam.sum())} truly damaged droplets:")
print(pd.DataFrame(rows).T.round(3).to_string())

🔍 **What just happened?** The damaged droplets form a clear tail of low counts, few genes and high mitochondrial
percentage, and the mitochondrial rule catches almost all of them by itself. Against the hidden truth every setting
finds every damaged droplet (recall 1.0); the thresholds differ in how many healthy cells are thrown away with them —
stricter thresholds (3 MADs) cost a few per cent precision, the lenient ones lose almost nothing. We keep the
5-MAD/3-MAD rule, which flags 226 droplets: all 211 damaged ones and 15 healthy cells. On real data the mitochondrial tail is less clean (some cell types, such as cardiomyocytes or
metabolically active cells, naturally have many mitochondrial transcripts), which is why thresholds should be checked per
sample and with an eye on which cell types they remove.

### Doublets: simulate them, then look for neighbours

A doublet's expression profile is roughly the sum of two cells. A heterotypic doublet (two different cell types) looks
like a hybrid that no real cell type produces. **Scrublet** (Wolock, Lopez & Klein 2019) turns this into a detector:

1. create **artificial doublets** by adding the counts of random pairs of observed droplets;
2. embed observed and artificial droplets together (normalise, log, PCA fitted on the observed cells);
3. for each observed droplet, the **doublet score** is the fraction of its $k$ nearest neighbours that are artificial
   doublets.

> 🔗 **Builds on** [Module 08 · k-NN](08_knn_naive_bayes.ipynb): this is a k-NN classifier of "real vs artificial
> doublet", trained on data we generated ourselves — a trick also known as *simulation-based* or *self-supervised*
> labelling.

A droplet surrounded by artificial doublets is probably a doublet itself. Two cells of the *same* type (homotypic
doublets — about $\sum_t p_t^2 \approx 22\%$ of all doublets with our proportions) have no hybrid profile; they look like
an ordinary cell with more UMIs, and profile-based detectors are generally considered blind to them.

In [ ]:
# ── 2.3 A Scrublet-style doublet detector ──
def doublet_scores(counts, rng, n_genes=500, n_pcs=10, k=30):
    """Fraction of artificial doublets among each droplet's k nearest neighbours (observed and artificial scores,
    and the pairs of droplets that were added to make the artificial doublets)."""
    pairs = rng.integers(0, len(counts), (len(counts), 2))
    artificial = counts[pairs[:, 0]] + counts[pairs[:, 1]]                  # step 1: add two random droplets
    L_obs, L_art = log_normalize(counts), log_normalize(artificial)
    genes = np.argsort(L_obs.var(axis=0))[-n_genes:]                         # most variable genes
    centre = L_obs[:, genes].mean(axis=0)
    _, _, Vt = np.linalg.svd(L_obs[:, genes] - centre, full_matrices=False)  # step 2: PCA of observed cells only
    P_obs, P_art = (L_obs[:, genes] - centre) @ Vt[:n_pcs].T, (L_art[:, genes] - centre) @ Vt[:n_pcs].T
    both = np.vstack([P_obs, P_art])
    is_art = np.r_[np.zeros(len(P_obs)), np.ones(len(P_art))]
    nn = NearestNeighbors(n_neighbors=k + 1).fit(both)
    score_obs = is_art[nn.kneighbors(P_obs)[1][:, 1:]].mean(axis=1)        # step 3: neighbour fractions
    score_art = is_art[nn.kneighbors(P_art)[1][:, 1:]].mean(axis=1)
    return score_obs, score_art, pairs


def otsu_threshold(x, bins=50):
    """Threshold that best separates a bimodal sample into two groups (maximal between-group variance)."""
    h, edges = np.histogram(x, bins=bins, range=(0, 1))
    mids, best, thr = (edges[:-1] + edges[1:]) / 2, -1.0, 0.5
    for i in range(1, bins):
        w0, w1 = h[:i].sum(), h[i:].sum()
        if w0 and w1:
            between = w0 * w1 * ((h[:i] * mids[:i]).sum() / w0 - (h[i:] * mids[i:]).sum() / w1) ** 2
            if between > best:
                best, thr = between, edges[i]
    return thr


t0 = time.time()
intact = ~damaged_call                                   # run the detector on the droplets that passed 2.1
score, score_art, pairs = doublet_scores(counts[intact], np.random.default_rng(SEED + 3))
thr = otsu_threshold(score_art)
print(f"doublet scores computed in {time.time() - t0:.1f} s; threshold from the artificial doublets' scores: {thr:.2f}")

In [ ]:
# ── 2.4 🧪 How good is the doublet detector? ──
is_dbl = obs.is_doublet.values[intact]
fig, ax = plt.subplots(figsize=(8, 3.4))
bins = np.linspace(0, 1, 31)
ax.hist(score_art, bins=bins, density=True, histtype="step", lw=2, color="0.3", label="artificial doublets")
ax.hist(score[~is_dbl], bins=bins, density=True, alpha=0.5, color=CLASS_COLORS[0], label="observed singlets (hidden truth)")
ax.hist(score[is_dbl], bins=bins, density=True, alpha=0.6, color=CLASS_COLORS[1], label="observed doublets (hidden truth)")
ax.axvline(thr, color="k", ls="--", label=f"threshold {thr:.2f}")
ax.set_xlabel("doublet score = fraction of artificial doublets among 30 neighbours"); ax.set_ylabel("density")
ax.set_title("SIMULATED doublet scores"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

rows = {}
for name, t in [("0.5", 0.5), (f"{thr:.2f} (Otsu on artificial doublets)", thr), ("top 5 % of droplets", np.quantile(score, 0.95))]:
    p, r, f = precision_recall_f1(is_dbl.astype(int), (score > t).astype(int))
    rows[f"score > {name}"] = {"called": int((score > t).sum()), "precision": p, "recall": r, "F1": f}
print(f"🧪 {int(is_dbl.sum())} true doublets among {intact.sum()} droplets:")
print(pd.DataFrame(rows).T.round(3).to_string())
tt = obs.true_type.to_numpy(str)[intact]
homo = tt[pairs[:, 0]] == tt[pairs[:, 1]]                  # artificial doublets made of two cells of one type
print(f"artificial doublets: {homo.mean():.0%} homotypic; scored above the threshold: {np.mean(score_art[homo] > thr):.0%} "
      f"of homotypic vs {np.mean(score_art[~homo] > thr):.0%} of heterotypic ones")

🔍 **What just happened?** Most artificial doublets score close to 1 (their neighbours are other artificial doublets);
the rest are spread thinly over lower scores, and Otsu's method puts the automatic threshold at 0.70. The true doublets
among the observed droplets mostly sit at high scores and the singlets mostly near 0, but the two overlap. The table
shows the classic trade-off: the threshold 0.5 catches 84 % of the doublets but flags more than two singlets for every
doublet (precision 0.30); the automatic threshold catches 69 % with precision 0.49; calling a fixed 5 % of droplets is
similar. That is roughly what one should expect from a doublet detector — a useful filter, not an oracle.

The last line holds a surprise: the homotypic artificial doublets, which have no hybrid profile, are *also* mostly
flagged (61 % against 93 % of heterotypic ones). The giveaway is depth: a doublet has about twice the UMIs and fewer
zeros, and that difference leaks through log-normalisation into the PCA. The same leak explains why deep singlets are
confused with doublets. On real data, where library sizes vary for many reasons, this depth signal is much less
reliable. Production tools (Scrublet, scDblFinder, DoubletFinder) refine every step, but the principle is this one.

⚠️ **Do not over-clean.** Every QC filter can remove real biology: small quiescent cells look like "low counts", and
cells caught in a transition between two states can look like doublets. Always check *which* cells a filter removes.

Now we remove the flagged droplets and genes detected in fewer than three cells.

In [ ]:
# ── 2.5 Apply the filters ──
keep = np.flatnonzero(intact)[score <= thr]
C = counts[keep]
cells_obs = obs.iloc[keep].copy()
gene_ok = (C > 0).sum(axis=0) >= 3
C, genes, gvar = C[:, gene_ok], gene_names[gene_ok], var[gene_ok]
true_type = cells_obs.true_type.to_numpy(str)
print(f"kept {len(keep)} of {len(obs)} droplets and {gene_ok.sum()} of {len(gene_ok)} genes")
print("hidden truth among the kept droplets:", {k: int(cells_obs[k].sum()) for k in ["is_doublet", "is_damaged"]},
      "| removed healthy singlets:", int(np.sum(~obs.is_doublet.values & ~obs.is_damaged.values) - np.sum(~cells_obs.is_doublet & ~cells_obs.is_damaged)))

🔍 **What just happened?** We keep 3,535 droplets. No damaged droplet survived, 54 doublets slipped through (we will meet
some of them again as annotation errors), and 129 healthy singlets were lost, almost all to the doublet filter. Losing
3–4 % of good cells is usually acceptable; losing a whole rare population would not be, which is why filters should be
checked per cell type.

✏️ **Your turn.** In cell 2.3 call `doublet_scores(counts[intact], np.random.default_rng(SEED + 3), n_pcs=30)` and re-run
2.3–2.4; then try `k=10`. How do precision and recall change? Can you explain why more PCs might let more of the depth
signal into the neighbour search?

---
## 3 · Normalisation and feature selection

> 📍 **Where we are.** The cells are clean, but their library sizes differ ten-fold and the counts' variance grows with
> their mean (Module 26). Before measuring distances between cells we must put them on a comparable scale and choose
> the genes that carry the biological signal.

### Library-size normalisation and the log

The standard recipe (Scanpy's `normalize_total` + `log1p`, the toolkit's `log_normalize`) scales each cell to the same
total (here 10,000) and takes a logarithm:

$$x_{cg} = \log\!\Big(1 + 10^4\,\frac{y_{cg}}{\sum_{g'} y_{cg'}}\Big) .$$

*In words:* express each gene as "counts per 10,000 UMIs of this cell", then compress with the log; the "+1" (the
**pseudo-count**) keeps zeros at zero.

🧮 **Math refresher — why a log?** For a count with mean $\mu$ and NB variance $\mu + \alpha\mu^2$, the delta method
gives $\operatorname{Var}(\log y) \approx \operatorname{Var}(y)/\mu^2 = 1/\mu + \alpha$. *In words:* on the log scale the variance no
longer grows with the mean; for well-expressed genes it levels off at $\alpha$. The log is a **variance-stabilising
transformation** — only approximately, and badly for small counts, where the pseudo-count dominates.

### Analytic Pearson residuals

Lause, Berens & Kobak (2021) proposed a transformation that comes directly from the GLMs of Module 26. Fit the simplest
possible model to every gene — *no biology at all*, every cell expresses gene $g$ at the same proportion $p_g$ of its
UMIs, $\mu_{cg} = n_c\,p_g$ (with $n_c$ the cell's total UMIs; the MLE is $\hat p_g = \sum_c y_{cg} / \sum_{c,g} y_{cg}$) —
and report how surprising each count is under that model:

$$r_{cg} = \frac{y_{cg} - \hat\mu_{cg}}{\sqrt{\hat\mu_{cg} + \hat\mu_{cg}^2/\theta}}, \qquad \hat\mu_{cg} = \frac{n_c \sum_{c'} y_{c'g}}{\sum_{c',g'} y_{c'g'}} .$$

*In words:* observed minus expected, divided by the NB standard deviation. A gene without biological variation has
residuals of variance ≈ 1 whatever its mean; a gene that differs between cell types has residual variance > 1. Lause
*et al.* use a fixed $\theta = 100$ (close to Poisson, reflecting technical noise) and clip residuals to $\pm\sqrt{N}$
($N$ = number of cells) so that a single huge count cannot dominate.

> 🔗 **Builds on** [Module 26](26_glm_count_data.ipynb), Section 2: $\log\mu_{cg} = \log n_c + \log p_g$ is a GLM with the
> library size as an **offset** and only an intercept per gene; the Pearson residual is its standardised residual.

In [ ]:
# ── 3.1 Log-normalisation and analytic Pearson residuals ──
def pearson_residuals(counts, theta=100.0):
    """Analytic Pearson residuals of the offset-only NB model (Lause, Berens & Kobak 2021), clipped at ±sqrt(N)."""
    Y = np.asarray(counts, float)
    mu = Y.sum(axis=1, keepdims=True) * Y.sum(axis=0, keepdims=True) / Y.sum()
    R = (Y - mu) / np.sqrt(mu + mu ** 2 / theta)
    return np.clip(R, -np.sqrt(len(Y)), np.sqrt(len(Y)))


L = log_normalize(C)                                   # cells × genes, log(1 + 10^4 · y / n_c)
R = pearson_residuals(C)
mean_raw = (C / C.sum(axis=1, keepdims=True) * 1e4).mean(axis=0)
is_marker = (gvar.marker_of != "").values              # hidden truth: marker gene of some cell type

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharex=True)
for ax, v, title in [(axes[0], (C / C.sum(axis=1, keepdims=True) * 1e4).var(axis=0), "normalised counts"),
                     (axes[1], L.var(axis=0), "log-normalised"), (axes[2], R.var(axis=0), "Pearson residuals")]:
    ax.scatter(mean_raw[~is_marker], v[~is_marker], s=4, color="0.6", alpha=0.6, label="other genes")
    ax.scatter(mean_raw[is_marker], v[is_marker], s=8, color=CLASS_COLORS[4], label="true marker genes")
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_title(f"variance of {title}"); ax.set_xlabel("mean (per 10⁴ UMIs)")
axes[2].axhline(1, color="k", ls="--", lw=1); axes[0].set_ylabel("variance across cells"); axes[0].legend(fontsize=7)
plt.suptitle("SIMULATED mean–variance relationships after three transformations", y=1.02)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Without a log (left), the variance simply tracks the mean — the most expressed genes look
the "most variable". After the log (middle), the relationship is bent but still strong: weakly expressed genes are
squeezed towards zero by the pseudo-count. The Pearson residuals (right) remove most of the mean dependence: the bulk of
genes sits near variance 1 (dashed line) — the variance expected without biology — and many marker genes (orange) stand
well above it. Some highly expressed non-marker genes also lie above 1, because the simulated cells are more variable
than the near-Poisson $\theta = 100$ assumes (the simulator uses gene-wise $\theta$ around 8).

### Highly variable genes (HVGs)

Most genes are uninformative for telling cell types apart; using all of them adds noise to every distance. We keep the
**highly variable genes**, by three criteria:

1. **raw variance of the log-normalised data** (the naïve choice);
2. **binned normalised dispersion** (Seurat v1 / Scanpy's `flavor="seurat"`): the log of variance/mean of the normalised
   counts, z-scored within 20 bins of genes with similar mean — "variable *compared with genes of the same expression
   level*";
3. **Pearson residual variance** (Lause *et al.* 2021).

Our ground truth knows which genes are cell-type markers, so we can score each criterion.

In [ ]:
# ── 3.2 🧪 Three ways to choose highly variable genes, graded by the hidden marker list ──
def binned_dispersion(C, n_bins=20):
    """Seurat-v1-style dispersion: log(var/mean) of normalised counts, z-scored within bins of similar mean."""
    E = C / C.sum(axis=1, keepdims=True) * 1e4
    m, v = E.mean(axis=0), E.var(axis=0)
    disp = np.log(v / m)
    bins = pd.qcut(np.log(m), n_bins, labels=False)
    z = np.zeros_like(disp)
    for b in range(n_bins):
        s = bins == b
        z[s] = (disp[s] - disp[s].mean()) / disp[s].std()
    return z


criteria = {"variance of log-normalised": L.var(axis=0), "binned normalised dispersion": binned_dispersion(C),
            "Pearson residual variance": R.var(axis=0)}
rows = {}
for name, s in criteria.items():
    order = np.argsort(s)[::-1]
    rows[name] = {f"markers among top {n}": is_marker[order[:n]].mean() for n in (100, 200, 500)}
    rows[name]["recall of all markers (top 500)"] = is_marker[order[:500]].sum() / is_marker.sum()
print(f"🧪 {is_marker.sum()} true marker genes among {len(genes)} ({is_marker.mean():.1%} of genes):")
print(pd.DataFrame(rows).T.round(3).to_string())
hvg = np.argsort(criteria["Pearson residual variance"])[::-1][:500]

🔍 **What just happened?** Choosing genes at random would give the baseline marker fraction of the header line (9 %).
The naïve log-variance criterion does better (26 % markers in its top 100, 15.5 % in its top 200) but it prefers genes
in a particular expression range (the hump of the middle panel above) rather than genes that differ between cell types.
Both mean-aware criteria do much better; on this simulation the binned dispersion ranks markers highest (58 % of its top
100; 75 % of all markers within its top 500), with the Pearson residuals second (38 %; 66 %). (Benchmarks on real data disagree about which
transformation is best for which task — Section 3's research box.) Remember that "true markers" here are only the genes
flagged by the simulator; genes shared by a whole lineage are informative too but not on the list, so these fractions
under-state the quality of all three criteria.

For the rest of the notebook we follow Lause *et al.*: the 500 genes with the largest Pearson residual variance, and the
residuals themselves as input to PCA.

✏️ **Your turn.** Change `theta=100.0` in `pearson_residuals` to `theta=8.0` (the simulator's typical value) and re-run
3.1–3.2. Does the residual-variance criterion improve? What would you need to know about a real data set to choose θ?

### 📄 Research connection — Best practices and the normalisation debate

- **What the field does.** How to transform UMI counts before PCA and clustering has been debated since droplet methods appeared. The
    shifted logarithm with a size factor is the default of Scanpy (Wolf, Angerer & Theis 2018) and Seurat; **sctransform**
    (Hafemeister & Satija 2019) uses Pearson residuals of a regularised NB GLM; Lause, Berens & Kobak (2021) showed that
    *analytic* residuals with a fixed overdispersion are simpler and perform as well. Ahlmann-Eltze & Huber (2023) compared
    many transformations and found that the shifted logarithm followed by PCA performed as well as or better than more
    sophisticated alternatives in their benchmarks. Community best-practice reviews (Luecken & Theis 2019; Heumos *et al.*
    2023) summarise the standard workflow.
- **What this notebook builds.** Log-normalisation, analytic Pearson residuals and three HVG criteria from scratch, graded by marker
    recovery against a hidden truth — a luxury real benchmarks lack (they must use proxies such as agreement with
    known cell labels).
- **Where the frontier is.** Open issues include transformations for very deep or very shallow data, count-based latent-variable
    models that skip transformation altogether (GLM-PCA, scVI in [Module 27](27_variational_inference_vae.ipynb)), and
    feature selection that is robust to batch effects ([Module 31](31_batch_integration.ipynb)). As of 2025 there is no
    single winner; the defensible choice is a standard pipeline plus checks that the conclusions do not depend on it.
- **Read next.** Heumos *et al.* 2023 (*Nature Reviews Genetics*), *Best practices for single-cell analysis across modalities*; Ahlmann-Eltze & Huber 2023 (*Nature Methods*), *Comparison of transformations for single-cell RNA-seq data*.

---
## 4 · Dimensionality reduction

> 📍 **Where we are.** Each cell is now a vector of 500 residuals. Distances in 500 dimensions are dominated by noise
> from hundreds of weakly informative genes. [Module 14](14_pca_dimensionality_reduction.ipynb) taught the remedy: keep
> the few directions of largest variance.

> 🔗 **Builds on** [Module 14 · PCA](14_pca_dimensionality_reduction.ipynb): centre, decompose with the SVD, keep the top
> components (scree plot), project. Here PCA also **denoises**: averaging many co-varying genes into one component
> cancels much of their independent counting noise.

In [ ]:
# ── 4.1 PCA of the Pearson residuals of the 500 HVGs ──
scores, components, evr = pca(R[:, hvg], n_components=50)
N_PCS = 15
Z = scores[:, :N_PCS]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1, 1.25]})
axes[0].plot(np.arange(1, 51), evr, "o-", ms=3, color="0.3")
axes[0].axvline(N_PCS + 0.5, color=CLASS_COLORS[1], ls="--", label=f"keep {N_PCS} PCs")
axes[0].set_yscale("log"); axes[0].set_xlabel("principal component"); axes[0].set_ylabel("fraction of variance")
axes[0].set_title("scree plot: a few PCs stand out of the noise"); axes[0].legend()
plot_embedding(scores[:, :2], true_type, ax=axes[1], s=3, title="PC1 vs PC2, coloured by the hidden true type (SIMULATED)")
plt.tight_layout(); plt.show()
print(f"fraction of the total variance of the 500 HVG residuals explained by PCs 1–5: {evr[:5].round(3).tolist()}; "
      f"by the first {N_PCS}: {evr[:N_PCS].sum():.2f}")

🔍 **What just happened?** The scree plot drops steeply over the first three components, more gently up to about the
sixth, and then flattens into a long tail of small, similar eigenvalues — the noise floor. PC1 and PC2 already separate
monocytes, B cells and NK cells; CD4 T, CD8 T and dendritic cells overlap in this projection and need further PCs. We
keep 15 components (29 % of the residual variance, most of it noise beyond the first few), comfortably past the elbow;
with six true types about five directions are needed to separate them, and a few extra PCs cost little. Choosing the number of PCs is a
model-selection decision ([Module 23](23_model_selection.ipynb)); in practice results are usually robust over a range.

### The k-nearest-neighbour graph

Almost every downstream step in Scanpy and Seurat — clustering, UMAP, trajectories — starts from a **k-NN graph**: each
cell is a node, joined by an edge to its $k$ nearest neighbours in PC space. The graph keeps only *local* information
("who resembles whom") and discards large distances, which in high dimensions are unreliable anyway.

In [ ]:
# ── 4.2 Build the k-NN graph (k = 15) and check its neighbourhoods against the truth ──
K_NN = 15
knn = NearestNeighbors(n_neighbors=K_NN + 1).fit(Z)
nbr = knn.kneighbors(Z)[1][:, 1:]                                  # drop each cell itself
graph = nx.Graph()
graph.add_nodes_from(range(len(Z)))
graph.add_edges_from(zip(np.repeat(np.arange(len(Z)), K_NN), nbr.ravel()))
purity = (true_type[nbr] == true_type[:, None]).mean()
print(f"k-NN graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges (mutual neighbours share one edge)")
print(f"🧪 fraction of each cell's {K_NN} neighbours with the same true type: {purity:.3f}")

🔍 **What just happened?** The graph has one node per cell and somewhat fewer than $15 \times$ (number of cells) edges,
because a pair of cells that are each other's neighbours shares one undirected edge. Almost all neighbours of a cell are
of the same true type (98.8 %) — the 15-dimensional PC space captures cell identity very well in this simulation. The
few mismatches are expected where types touch, such as the two T-cell types, and around surviving doublets.

### t-SNE for visualisation

To *look* at the data we need two dimensions. **t-SNE** (van der Maaten & Hinton 2008) places points in 2-D so that
each point's neighbours in the original space remain its neighbours: it turns distances into neighbour probabilities in
both spaces and minimises their Kullback–Leibler divergence (a measure of how different two probability distributions
are; [Module 27](27_variational_inference_vae.ipynb) treats it in depth), using a heavy-tailed Student-t kernel in 2-D so
that clusters spread out. It is excellent at showing local structure and **not** designed to preserve anything else. t-SNE costs time, so we
embed a random subsample of 1,500 cells.

In [ ]:
# ── 4.3 t-SNE of a subsample, and what it preserves (🧪) ──
sub = np.random.default_rng(SEED + 4).choice(len(Z), 1500, replace=False)
t0 = time.time()
emb = TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED, max_iter=500).fit_transform(Z[sub])
print(f"t-SNE of {len(sub)} cells in {time.time() - t0:.1f} s")

def shared_neighbours(A, B, k=15):
    """Average fraction of each point's k nearest neighbours in A that are also among its k nearest in B."""
    na = NearestNeighbors(n_neighbors=k + 1).fit(A).kneighbors(A)[1][:, 1:]
    nb = NearestNeighbors(n_neighbors=k + 1).fit(B).kneighbors(B)[1][:, 1:]
    return np.mean([len(set(a) & set(b)) / k for a, b in zip(na, nb)])

types_sub = true_type[sub]
cent_pca = np.array([Z[sub][types_sub == t].mean(axis=0) for t in PBMC_TYPES])
cent_tsne = np.array([emb[types_sub == t].mean(axis=0) for t in PBMC_TYPES])
iu = np.triu_indices(len(PBMC_TYPES), 1)
d_pca = np.linalg.norm(cent_pca[:, None] - cent_pca[None], axis=2)[iu]
d_tsne = np.linalg.norm(cent_tsne[:, None] - cent_tsne[None], axis=2)[iu]
print(f"🧪 15-NN preserved (PCA → t-SNE): {shared_neighbours(Z[sub], emb):.2f}; "
      f"random 2-D layout: {shared_neighbours(Z[sub], np.random.default_rng(1).normal(size=(len(sub), 2))):.2f}")
print(f"🧪 Spearman correlation of the 15 between-type centroid distances (PCA vs t-SNE): {stats.spearmanr(d_pca, d_tsne)[0]:.2f}")

In [ ]:
# ── 4.4 The embedding, coloured by true type and by condition ──
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
plot_embedding(emb, types_sub, ax=axes[0], s=3, title="t-SNE (SIMULATED), hidden true type")
plot_embedding(emb, cells_obs.condition.to_numpy(str)[sub], ax=axes[1], s=3, title="condition")
plot_embedding(emb, np.where(cells_obs.is_cycling.to_numpy(bool)[sub], "cycling", "not cycling"), ax=axes[2], s=3,
               title="cycling cells (hidden truth)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The cell types form well-separated islands. Stimulated and control cells separate completely
within monocytes, whose response is strongest (two separate islands), partly within the two T-cell types, and only
weakly within B and NK cells, where they look largely mixed. A
small extra island holds cycling T cells: in T cells, whose CD4/CD8 differences are subtle, the strong cell-cycle
programme outweighs cell identity (cycling monocytes or NK cells stay with their type). The 🧪 numbers say what the
picture can and cannot be trusted for: about half (0.49) of each cell's 15 PCA-neighbours are still among its 15
neighbours in 2-D, against 0.01 for a random layout — local structure is largely kept. But the *distances between
islands* are only loosely related to the distances between the same cell types in PC space (rank correlation 0.49):
t-SNE spreads clusters apart to make them visible, whatever their true separation.

⚠️ **What 2-D embeddings do not show.** Distances between clusters, cluster sizes (t-SNE expands dense clusters and
shrinks sparse ones) and the empty space between islands carry little meaning; different random seeds or perplexities
give different-looking maps. UMAP shares these limitations. Kobak & Berens (2019) give practical advice on making t-SNE
more faithful (PCA initialisation, perplexity choice); Chari & Pachter (2023) argue that such embeddings should not be
used for quantitative conclusions at all. Use them to *look*, and do every analysis in the PC space or on the graph.

✏️ **Your turn.** Re-run 4.3 with `perplexity=5` and with `perplexity=100`. How do the islands change? Which of the two
🧪 numbers changes more?

---
## 5 · Clustering

> 📍 **Where we are.** We have a PC space and a k-NN graph. Now we group the cells. [Module 12](12_clustering_kmeans_hierarchical.ipynb)
> and [Module 13](13_mixture_models_em.ipynb) gave us k-means and Gaussian mixtures; single-cell pipelines use
> **community detection on the k-NN graph** instead. Why?

> 🔗 **Builds on** [Module 12](12_clustering_kmeans_hierarchical.ipynb) (k-means minimises within-cluster squared
> distances; K must be chosen) and [Module 13](13_mixture_models_em.ipynb) (a GMM is a soft, elliptical k-means fitted by
> EM). Both are compared below with graph clustering on exactly the same 15 PCs.

A k-means cluster is a ball around a centre, and a GMM component an ellipse; both need the number of clusters $K$ and
struggle with clusters of very different sizes or curved shapes. A graph community is just a set of cells that are
**more connected to each other than expected** — any shape, any size, and the number of communities comes out of the
optimisation.

🧮 **Math refresher — modularity.** Let $A_{ij} = 1$ if cells $i$ and $j$ are joined by an edge, $k_i$ the degree of
cell $i$ (its number of edges) and $m$ the total number of edges. For a partition into communities $c_i$,

$$Q = \frac{1}{2m}\sum_{i,j}\Big[A_{ij} - \gamma\,\frac{k_i k_j}{2m}\Big]\,\delta(c_i, c_j) ,$$

where $\delta(c_i, c_j) = 1$ if $i$ and $j$ are in the same community. *In words:* count the edges that fall inside
communities and subtract the number expected if edges were placed at random between cells with the same degrees;
$Q$ is high when communities are much denser inside than chance. The **resolution** $\gamma$ scales the penalty: larger
$\gamma$ makes large communities "expensive" and yields more, smaller clusters.

**Louvain** (Blondel *et al.* 2008) maximises $Q$ greedily: move single cells to the neighbouring community that
increases $Q$ most, then merge each community into a super-node and repeat. **Leiden** (Traag, Waltman & van Eck 2019)
adds a refinement step that guarantees well-connected communities and is the method Scanpy recommends; NetworkX
provides Louvain, which we use.

In [ ]:
# ── 5.1 Louvain at several resolutions, k-means and a Gaussian mixture — 🧪 ARI against the true types ──
def louvain_labels(graph, resolution, seed=SEED):
    """Louvain communities as integer labels, numbered from the largest community down."""
    comms = sorted(nx.community.louvain_communities(graph, resolution=resolution, seed=seed), key=len, reverse=True)
    lab = np.empty(graph.number_of_nodes(), int)
    for c, members in enumerate(comms):
        lab[list(members)] = c
    return lab, nx.community.modularity(graph, comms)          # standard modularity (γ = 1), comparable across runs


t0 = time.time()
results, louvain = {}, {}
for res in [0.2, 0.5, 1.0, 2.0]:
    louvain[res], Q = louvain_labels(graph, res)
    results[f"Louvain, resolution {res}"] = {"clusters": louvain[res].max() + 1, "modularity (γ = 1)": Q,
                                             "ARI": adjusted_rand_score(true_type, louvain[res])}
km = KMeans(n_clusters=6, n_init=10, random_state=SEED).fit_predict(Z)
gmm = GaussianMixture(n_components=6, n_init=3, random_state=SEED).fit(Z).predict(Z)
for name, lab_ in [("k-means, K = 6", km), ("GMM (full covariance), K = 6", gmm)]:
    comms_ = [set(np.flatnonzero(lab_ == c)) for c in range(6)]
    results[name] = {"clusters": 6, "modularity (γ = 1)": nx.community.modularity(graph, comms_),
                     "ARI": adjusted_rand_score(true_type, lab_)}
print(f"🧪 clustering of {len(Z)} cells ({time.time() - t0:.1f} s):")
print(pd.DataFrame(results).T.round(3).to_string())

In [ ]:
# ── 5.2 Which types merge or split? Contingency tables (rows: hidden truth) ──
type_cond = pd.Series(true_type) + np.where(cells_obs.condition.to_numpy(str) == "stimulated", " (stim)", " (ctrl)")
print("k-means, K = 6:\n", pd.crosstab(pd.Series(true_type, name="true type"), pd.Series(km, name="k-means cluster")).to_string())
lab = louvain[0.5]
print("\nLouvain, resolution 0.5 (true type split by condition):\n",
      pd.crosstab(type_cond.rename("true type (condition)"), pd.Series(lab, name="Louvain cluster")).to_string())

🔍 **What just happened?** Read the two outputs together.

- **k-means with the "right" K = 6 is wrong in an instructive way.** It spends two of its six clusters on the monocytes
  (380 + 365 cells: control and stimulated monocytes differ strongly — Section 7) and has none left for the 76 dendritic
  cells, which it merges into the CD8 T cluster (68) and the CD4 T cluster (8). k-means minimises within-cluster squared
  distance, so splitting a large, spread-out group pays off more than isolating a tiny one. The GMM makes a similar
  trade-off (ARI 0.90 and 0.91).
- **Louvain at low resolution** (0.2 and 0.5 give the same partition here) finds seven communities: every true type,
  including the dendritic cells (cluster 6), plus a split of the monocytes into a control (cluster 4) and a stimulated
  (cluster 3) community. Its ARI of 0.92 is below 1 only because of that split — which is real biology (a *state*), not
  an error.
- **Higher resolution** keeps cutting the data into smaller pieces (11 and 15 communities) and the ARI against the six
  types falls (0.56 and 0.43). Yet the standard modularity is *highest* for the 11-community partition (0.794 vs 0.771):
  modularity measures how community-like a partition is, not whether it matches the biology you care about.

This is the central lesson of single-cell clustering: **a cluster is a hypothesis, not a cell type.** Clusters can be
cell types, states within a type (stimulated monocytes), cell-cycle phases, doublets or technical groups. The resolution
is a choice about granularity, made with biological knowledge and checked with marker genes (Section 6). Note also that
the ARI needs the truth: on real data we can only check stability (e.g. clustering subsamples) and marker coherence.

✏️ **Your turn.** (1) Run k-means with `n_clusters=7` and `8`. Does it ever isolate the dendritic cells? (2) Cross-tabulate
`louvain[2.0]` against `cells_obs.is_cycling`. Do the cycling cells form a cluster of their own at high resolution?

In [ ]:
# ── 5.3 The clusters on the t-SNE map ──
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
plot_embedding(emb, true_type[sub], ax=axes[0], s=3, legend=False, title="hidden true type")
plot_embedding(emb, np.array([f"k{c}" for c in km[sub]]), ax=axes[1], s=3, legend=False, title="k-means, K = 6")
plot_embedding(emb, np.array([f"L{c}" for c in lab[sub]]), ax=axes[2], s=3, legend=False, title="Louvain, resolution 0.5")
for ax, labels in zip(axes, [true_type[sub], np.array([f"k{c}" for c in km[sub]]), np.array([f"L{c}" for c in lab[sub]])]):
    for l in np.unique(labels):                                    # write each label at its group's median position
        xy = np.median(emb[labels == l], axis=0)
        ax.text(*xy, l, fontsize=7, ha="center", weight="bold")
plt.suptitle("SIMULATED: the same t-SNE map coloured by three partitions", y=1.0)
plt.tight_layout(); plt.show()

### 📄 Research connection — Graph-based clustering of cells

- **What the field does.** Graph community detection became the standard for single-cell data with PhenoGraph (Levine *et al.* 2015,
    on mass cytometry) and the Louvain implementation in Seurat; Scanpy (Wolf, Angerer & Theis 2018) recommends Leiden
    (Traag, Waltman & van Eck 2019), which fixes Louvain's occasional badly connected communities, and Seurat (Hao *et
    al.* 2021) offers both. Louvain itself (Blondel *et al.* 2008) was designed for networks with millions of nodes.
- **What this notebook builds.** A k-NN graph in PC space, modularity explained and Louvain applied at four resolutions, compared with
    k-means and Gaussian mixtures by the ARI against the hidden truth. We saw graph clustering isolate a rare cell type
    that k-means merged, and over-clustering at high resolution.
- **Where the frontier is.** Choosing the resolution and judging over-clustering remain open problems: stability-based criteria,
    hierarchical and multi-resolution annotation, and statistical tests that account for the clustering having been
    learned from the same data ("double dipping") are active topics. Reference-based annotation
    ([Module 31](31_batch_integration.ipynb)) increasingly replaces de-novo clustering for well-studied tissues.
- **Read next.** Traag, Waltman & van Eck 2019 (*Scientific Reports*), *From Louvain to Leiden: guaranteeing well-connected communities*; Wolf, Angerer & Theis 2018 (*Genome Biology*), *SCANPY: large-scale single-cell gene expression data analysis*.

---
## 6 · Marker genes and annotation

> 📍 **Where we are.** We have seven numbered clusters. To give them names we ask which genes distinguish each cluster
> from all other cells, and compare those genes with what immunologists know.

The workhorse is the **Wilcoxon rank-sum test** (Mann–Whitney U test), the default of Scanpy's `rank_genes_groups` and
Seurat's `FindMarkers`. For one gene, rank all $n$ cells by expression; if the cluster (size $n_1$) had no special
expression, its ranks would be a random subset. The statistic is

$$U = \sum_{i \in \text{cluster}} \operatorname{rank}_i - \frac{n_1(n_1 + 1)}{2}, \qquad \mathbb E[U] = \frac{n_1 n_2}{2}, \qquad
\operatorname{Var}[U] = \frac{n_1 n_2}{12}\Big[(n + 1) - \frac{\sum_t (t^3 - t)}{n(n-1)}\Big] .$$

*In words:* $U$ counts how often a cluster cell beats an outside cell; under "no difference" it is centred at half of all
$n_1 n_2$ comparisons. The **tie correction** (the sum over groups of $t$ tied values) matters enormously here, because
most cells share the value 0. For thousands of cells, $z = (U - \mathbb E U)/\sqrt{\operatorname{Var} U}$ is standard normal.
Ranking all cells once per gene lets us test every cluster with one sum.

In [ ]:
# ── 6.1 A vectorised rank-sum test (all genes, one cluster vs the rest), checked against SciPy ──
ranks = stats.rankdata(L, axis=0)                                     # rank cells within each gene (ties → average)
n_cells = len(L)
tie_term = np.array([np.sum(c ** 3 - c) for c in (np.unique(L[:, j], return_counts=True)[1] for j in range(L.shape[1]))])

def rank_sum_test(in_group):
    """z-scores and two-sided p-values of the Wilcoxon rank-sum test of every gene: cells in_group vs the rest."""
    n1 = in_group.sum(); n2 = n_cells - n1
    U = ranks[in_group].sum(axis=0) - n1 * (n1 + 1) / 2
    sd = np.sqrt(n1 * n2 / 12 * ((n_cells + 1) - tie_term / (n_cells * (n_cells - 1))))
    z = (U - n1 * n2 / 2) / sd
    return z, 2 * stats.norm.sf(np.abs(z))

z0, p0 = rank_sum_test(lab == 0)
ref = stats.mannwhitneyu(L[lab == 0][:, :4], L[lab != 0][:, :4], axis=0, use_continuity=False, method="asymptotic")
print("our p-values: ", p0[:4]); print("SciPy:        ", ref.pvalue)

marker_table = {}
for c in range(lab.max() + 1):
    z, p = rank_sum_test(lab == c)
    top5 = np.argsort(-z)[:5]
    marker_table[f"cluster {c} ({(lab == c).sum()} cells)"] = [f"{genes[j]}" + (f" [{gvar.marker_of.iloc[j]}]" if gvar.marker_of.iloc[j] else "") for j in top5]
print("\ntop 5 up-regulated genes per cluster [hidden truth: marker of …]:")
print(pd.DataFrame(marker_table, index=[f"#{i + 1}" for i in range(5)]).T.to_string())

🔍 **What just happened?** Our 10-line test reproduces SciPy's p-values exactly. The top-ranked genes of every cluster
are dominated by true markers of one cell type (in square brackets, from the hidden truth), and several of them carry the
canonical names an immunologist would recognise: IL7R and CD3 genes, CD8A/CD8B, NKG7/GNLY/KLRD1/PRF1, MS4A1/CD79A/CD19,
LYZ/S100A8/S100A9/CD14, FCER1A/CST3/CLEC10A. The two monocyte clusters share their top markers.

⚠️ **Double dipping.** These p-values are astronomically small, but they are *not* valid tests of "is this cluster
different?": the clusters were defined from the same expression data, so some genes are guaranteed to differ. Marker
p-values are useful for **ranking** genes, not as evidence that clusters are real (Section 5's research box).

### A dot plot and an annotation by marker scores

The **dot plot** summarises canonical markers per cluster: dot size = fraction of cells with non-zero expression, colour
= mean expression. Then we annotate automatically: for each cell type, average the z-scored expression of its canonical
markers (a **marker score**), average the score per cluster, and give each cluster the name of its highest score.

In [ ]:
# ── 6.2 Dot plot of canonical markers ──
canon = [g for _, m, _ in PBMC_TYPES.values() for g in m if g in set(genes)]
canon = list(dict.fromkeys(canon))                                     # unique, in the order of PBMC_TYPES
gi = [int(np.flatnonzero(genes == g)[0]) for g in canon]
clusters = np.arange(lab.max() + 1)
frac = np.array([(C[lab == c][:, gi] > 0).mean(axis=0) for c in clusters])
mexp = np.array([L[lab == c][:, gi].mean(axis=0) for c in clusters])
mexp_scaled = mexp / mexp.max(axis=0)                                  # scale each gene to its maximum

fig, ax = plt.subplots(figsize=(11, 3.4))
xx, yy = np.meshgrid(np.arange(len(canon)), clusters)
sc = ax.scatter(xx.ravel(), yy.ravel(), s=200 * frac.ravel(), c=mexp_scaled.ravel(), cmap="Reds", edgecolor="0.4", lw=0.3)
ax.set_xticks(range(len(canon))); ax.set_xticklabels(canon, rotation=90, fontsize=8)
ax.set_yticks(clusters); ax.set_yticklabels([f"cluster {c}" for c in clusters], fontsize=8)
ax.set_xlim(-0.7, len(canon) - 0.3); ax.set_ylim(len(clusters) - 0.4, -0.6)
plt.colorbar(sc, ax=ax, label="mean expression (scaled)", shrink=0.8)
ax.set_title("SIMULATED dot plot: dot size = fraction of cells expressing, colour = mean expression")
plt.tight_layout(); plt.show()

In [ ]:
# ── 6.3 Annotate clusters by marker scores — 🧪 accuracy against the hidden truth ──
Lz = (L - L.mean(axis=0)) / L.std(axis=0)
gene_pos = {g: j for j, g in enumerate(genes)}
cell_scores = pd.DataFrame({t: Lz[:, [gene_pos[g] for g in m if g in gene_pos]].mean(axis=1) for t, (_, m, _) in PBMC_TYPES.items()})
cluster_scores = cell_scores.groupby(lab).mean()
annotation = cluster_scores.idxmax(axis=1)
print("mean marker score per cluster (rows) and cell type (columns):")
print(cluster_scores.round(2).to_string())
print("\nannotation:", annotation.to_dict())

cell_type = annotation.values[lab]                                     # predicted type of every cell
print(f"\n🧪 annotation accuracy (cells whose predicted type equals the hidden true type): {np.mean(cell_type == true_type):.3f}")
print(pd.crosstab(pd.Series(true_type, name="true type"), pd.Series(cell_type, name="annotated as")).to_string())

🔍 **What just happened?** Each cluster has one clearly highest marker score, and the annotation names both monocyte
clusters "Monocyte" — merging the state split back into one type. 99.4 % of cells receive their true type. The 21 errors
include confusions between the two T-cell types (9 cells), whose canonical markers overlap (both express CD3E and
CD3D), and cells such as surviving doublets, whose "true type" is that of only one of their two cells. The CD4 vs CD8
scores are also the least contrasted rows of the table, as in real data, where CD4 and CD8 T cells are notoriously hard
to separate from RNA alone (the CD4 transcript itself is detected in few cells — see its tiny dots in the dot plot).

On real data the canonical markers are known only for well-studied types and conditions; automated annotation tools
transfer labels from annotated **reference atlases** instead (CellTypist, Azimuth, scANVI — [Module 31](31_batch_integration.ipynb)).

✏️ **Your turn.** Annotate the clusters of `louvain[2.0]` with the same marker-score rule. Is the accuracy still high?
What does that tell you about over-clustering followed by annotation?

### 📄 Research connection — Cell atlases and cell-type annotation

- **What the field does.** The **Human Cell Atlas** (Regev *et al.* 2017) set out to map every cell type of the human body; droplet
    methods (e.g. Zheng *et al.* 2017, the 10x Genomics platform and its PBMC data) made atlases of millions of cells
    possible, and portals such as CELLxGENE now serve tens of millions of annotated cells. Annotation moved from manual
    marker inspection of clusters to automated label transfer from references (Seurat's reference mapping in Hao *et al.*
    2021; CellTypist; scANVI).
- **What this notebook builds.** The manual workflow from scratch: rank-sum marker tests (checked against SciPy), a dot plot of canonical
    markers, and marker-score annotation — with an accuracy measured against the hidden truth.
- **Where the frontier is.** Consistent cell-type nomenclature across atlases (cell ontologies), rare and transitional states that do
    not fit discrete labels, annotation under disease or perturbation (where references may lack the relevant states),
    and foundation models pre-trained on atlases ([Module 43](43_llms_ai_agents.ipynb)) whose benefits over simple
    baselines are still being evaluated.
- **Read next.** Regev *et al.* 2017 (*eLife*), *The Human Cell Atlas*; Hao *et al.* 2021 (*Cell*), *Integrated analysis of multimodal single-cell data*.

---
## 7 · Comparing conditions

> 📍 **Where we are.** Every cell now has a type. The biological question of Kang *et al.* was how each type responds to
> interferon. [Module 26](26_glm_count_data.ipynb), Section 7, showed on monocytes that the donor is the unit of
> replication; here we run the comparison for **all** annotated types and grade it with the truth.

For each cell type we compare two analyses:

- **cell-level Wilcoxon test** — stimulated vs control *cells* (what a naïve call of Scanpy's `rank_genes_groups` or
  Seurat's `FindMarkers` with the condition as grouping does);
- **pseudobulk NB GLM** — sum the counts of each donor's cells of that type (six samples) and fit the empirical-Bayes NB
  GLM of Module 26.

The truth (`true_logfc`): the 10 interferon-stimulated genes respond in every type, and 40 further genes respond only in
monocytes. The next cell re-implements the Module 26 machinery compactly.

> 🔗 **Builds on** [Module 26](26_glm_count_data.ipynb), Sections 2–7: IRLS for the NB GLM with offsets, median-of-ratios
> size factors, Cox–Reid dispersions shrunk to a mean trend (empirical Bayes) and the Wald test — here condensed into
> two cells.

In [ ]:
# ── 7.1 A compact negative-binomial GLM with empirical-Bayes dispersions (from Module 26) ──
def nb_irls(Y, X, offset, alpha, n_iter=30):
    """Log-link NB GLM for every row of Y (genes × samples) by IRLS; returns beta, standard errors, fitted means."""
    beta = np.zeros((len(Y), X.shape[1])); beta[:, 0] = np.log((Y / np.exp(offset)).mean(1) + 0.1)
    for _ in range(n_iter):
        eta = np.clip(beta @ X.T + offset, -30, 30); mu = np.exp(eta)
        W = mu / (1 + alpha[:, None] * mu)
        beta = np.linalg.solve(np.einsum("np,gn,nq->gpq", X, W, X),
                               np.einsum("np,gn->gp", X, W * (eta - offset + (Y - mu) / mu))[..., None])[..., 0]
    mu = np.exp(np.clip(beta @ X.T + offset, -30, 30)); W = mu / (1 + alpha[:, None] * mu)
    return beta, np.sqrt(np.einsum("gpp->gp", np.linalg.inv(np.einsum("np,gn,nq->gpq", X, W, X)))), mu

def nb_cr_loglik(Y, mu, alpha, X):
    """Cox–Reid adjusted NB log-likelihood of the dispersion, per gene."""
    th = 1 / alpha[:, None]
    ll = (special.gammaln(Y + th) - special.gammaln(th) - special.gammaln(Y + 1) + th * np.log(th / (th + mu))
          + Y * np.log(mu / (th + mu))).sum(1)
    W = mu / (1 + alpha[:, None] * mu)
    return ll - 0.5 * np.linalg.slogdet(np.einsum("np,gn,nq->gpq", X, W, X))[1]

def golden_max(f, lo, hi, n, n_iter=50):
    """Vectorised golden-section search: maximise f (one value per gene) over [lo, hi]."""
    a, b, r = np.full(n, lo), np.full(n, hi), (np.sqrt(5) - 1) / 2
    for _ in range(n_iter):
        c, d = b - r * (b - a), a + r * (b - a)
        left = f(c) > f(d); b, a = np.where(left, d, b), np.where(left, a, c)
    return (a + b) / 2

In [ ]:
# ── 7.2 Pseudobulk differential expression: size factors, shrunken dispersions, Wald test ──
def median_of_ratios(Y):
    ok = (Y > 0).all(axis=1)
    return np.exp(np.median(np.log(Y[ok]) - np.log(Y[ok]).mean(axis=1, keepdims=True), axis=0))

def pseudobulk_de(Y, X, coef=1):
    """Module 26's mini-DESeq2: returns log2 fold changes and Wald p-values of column `coef` of X."""
    off, (n, p), G = np.log(median_of_ratios(Y)), X.shape, len(Y)
    a = np.full(G, 0.1)
    for _ in range(3):                                                  # Cox–Reid gene-wise dispersions
        mu = nb_irls(Y, X, off, a)[2]
        a = np.exp(golden_max(lambda la: nb_cr_loglik(Y, mu, np.exp(la), X), np.log(1e-8), np.log(10.0), G))
    mean = (Y / np.exp(off)).mean(axis=1); use = a > 1e-6
    T = np.column_stack([np.ones(use.sum()), 1 / mean[use]]); c = np.array([0.05, 1.0])
    for _ in range(10):                                                 # gamma-GLM trend a0 + a1/mean
        w = 1 / (T @ c) ** 2; c = np.maximum(np.linalg.solve(T.T @ (w[:, None] * T), T.T @ (w * a[use])), 1e-6)
    trend = c[0] + c[1] / mean
    v_prior = max(stats.median_abs_deviation(np.log(a[use] / trend[use]), scale="normal") ** 2
                  - special.polygamma(1, (n - p) / 2), 0.25)
    mu = nb_irls(Y, X, off, trend)[2]                                   # empirical-Bayes MAP dispersions
    a_map = np.exp(golden_max(lambda la: nb_cr_loglik(Y, mu, np.exp(la), X) - (la - np.log(trend)) ** 2 / (2 * v_prior),
                              np.log(1e-8), np.log(10.0), G))
    beta, se, _ = nb_irls(Y, X, off, a_map)
    return beta[:, coef] / np.log(2), 2 * stats.norm.sf(np.abs(beta[:, coef] / se[:, coef]))

In [ ]:
# ── 7.3 🧪 Every annotated cell type: cell-level Wilcoxon vs pseudobulk, graded by true_logfc ──
donor, cond = cells_obs.donor.to_numpy(str), cells_obs.condition.to_numpy(str)
donors = sorted(set(donor))
stim_donor = np.array([cond[donor == d][0] == "stimulated" for d in donors], float)
X_pb = np.column_stack([np.ones(len(donors)), stim_donor])
true_lfc = scd["true_logfc"][genes]
rows, est_l2fc = [], {}
t0 = time.time()
for ct in PBMC_TYPES:
    m = cell_type == ct
    PB = np.column_stack([C[m & (donor == d)].sum(axis=0) for d in donors]).astype(float)   # genes × donors
    tested = PB.sum(axis=1) >= 10
    l2, p_pb = pseudobulk_de(PB[tested], X_pb)
    s = cond[m] == "stimulated"
    p_w = stats.mannwhitneyu(L[m][s][:, tested], L[m][~s][:, tested], axis=0).pvalue
    responds = true_lfc.loc[ct].values[tested] != 0
    for method, p in [("cell-level Wilcoxon", p_w), ("pseudobulk NB GLM", p_pb)]:
        hit = benjamini_hochberg(p) < 0.05
        rows.append({"cell type": ct, "cells": int(m.sum()), "method": method, "true responders": int(responds.sum()),
                     "discoveries": int(hit.sum()), "false": int(np.sum(hit & ~responds)),
                     "realised FDR": np.sum(hit & ~responds) / max(hit.sum(), 1), "power": np.sum(hit & responds) / responds.sum()})
    est_l2fc[ct] = pd.Series(l2, index=genes[tested])
de_table = pd.DataFrame(rows)
print(f"🧪 BH q < 0.05 in every annotated cell type ({time.time() - t0:.1f} s):")
print(de_table.round(3).to_string(index=False))
print("\ntotals:", de_table.groupby("method")[["discoveries", "false"]].sum().to_dict("index"))

In [ ]:
# ── 7.4 Estimated (pseudobulk) vs true log2 fold changes of the responding genes ──
ifn_genes = [g for g in genes if gvar.loc[g, "is_interferon"]]
mono_resp = [g for g in genes if true_lfc.loc["Monocyte", g] != 0 and g not in ifn_genes][:20]
show = ifn_genes + mono_resp
true_mat = (true_lfc.loc[list(PBMC_TYPES), show] / np.log(2)).values
est_mat = np.array([[est_l2fc[t].get(g, np.nan) for g in show] for t in PBMC_TYPES])
fig, axes = plt.subplots(2, 1, figsize=(12, 4.6), sharex=True)
for ax, M, title in [(axes[0], true_mat, "TRUE log2 fold change (hidden)"), (axes[1], est_mat, "ESTIMATED by pseudobulk NB GLM")]:
    im = ax.imshow(M, cmap="RdBu_r", vmin=-3, vmax=3, aspect="auto")
    ax.set_yticks(range(len(PBMC_TYPES))); ax.set_yticklabels(list(PBMC_TYPES), fontsize=8); ax.set_title(title, fontsize=9)
axes[1].set_xticks(range(len(show))); axes[1].set_xticklabels(show, rotation=90, fontsize=7)
axes[0].axvline(len(ifn_genes) - 0.5, color="k", lw=1); axes[1].axvline(len(ifn_genes) - 0.5, color="k", lw=1)
fig.colorbar(im, ax=axes, shrink=0.8, label="log2 fold change (stim / ctrl)")
plt.suptitle("SIMULATED: interferon genes (left of the line) respond everywhere, the monocyte programme only in monocytes", y=1.0)
plt.show()

🔍 **What just happened?** The table repeats Module 26's lesson in every cell type at once. Over all six types the
cell-level Wilcoxon test makes 536 discoveries, 460 of them false; the more cells a type has, the more it "finds" (CD4 T
cells, the largest population, give 179 calls for 10 true responders), because more cells make tiny donor-to-donor
differences look certain. The pseudobulk test makes 62 discoveries with 5 false ones, at the price of lower power in
small populations: it finds all 10 interferon genes in CD4 T cells but only 4 in B cells, and with 77 dendritic cells
split over six donors nothing can be called. The heat maps show that the pseudobulk fold changes reproduce the biology:
the interferon genes go up in every type, the monocyte-specific programme (up and down) appears in monocytes only —
exactly the structure of the truth. (White cells: genes too weakly expressed in that type to be tested.)

Not all of the cell-level "false discoveries" come from donor differences. Each cell has a fixed UMI budget, so when the
responding genes go up, the unchanged genes' *share* of the reads falls slightly in stimulated cells — by a few per cent
(about 4 % for monocytes in Module 26, Section 7, where it is measured). Median-of-ratios size factors largely absorb
this compositional shift in the pseudobulk, but with hundreds of cells a cell-level test on log-normalised data can
detect it. It is a real effect of sequencing data, not of our simulator alone — and biologically meaningless.

⚠️ **Errors propagate.** The DE analysis used *our* annotation, not the truth. Mis-annotated cells (or doublets that
survived QC) contaminate the pseudobulk of the wrong type; a strong response in monocytes can then leak into, say, the
dendritic-cell pseudobulk. Clean annotation is a prerequisite of cell-type-specific DE.

✏️ **Your turn.** Repeat the pseudobulk analysis for monocytes with the *true* types (`true_type == "Monocyte"`) instead
of the annotation. Does anything change? Then merge CD4 and CD8 T cells into one "T" pseudobulk: what happens to power
and why?

---
## 8 · Composition

> 📍 **Where we are.** Stimuli and diseases change not only what cells express but *which* cells are present. Here we
> count cell types per donor, provoke a change on purpose, and meet the trap hidden in proportions.

Our simulation has the same expected proportions in every donor and condition, so first we check that we see no change.
Then we plant one: we remove half of the NK cells from the stimulated donors (as if the stimulus killed them). In
**absolute** numbers only NK cells change. But a single-cell experiment captures a fixed number of cells per sample, so
we observe **proportions**, which must sum to 1: if NK cells drop, every other type's share *rises*, although nothing
happened to it. Proportions are **compositional** data.

In [ ]:
# ── 8.1 Proportions per donor, before and after a planted NK-cell loss ──
def type_proportions(donor, cell_type):
    """Donor × cell-type table of proportions."""
    tab = pd.crosstab(donor, cell_type)
    return tab.div(tab.sum(axis=1), axis=0)

rng = np.random.default_rng(SEED + 9)
lost = (cell_type == "NK") & (cond == "stimulated") & (rng.random(len(cell_type)) < 0.5)    # the planted change
prop_before = type_proportions(donor, cell_type)
prop_after = type_proportions(donor[~lost], cell_type[~lost])
is_stim = np.isin(prop_before.index, [d for d, s in zip(donors, stim_donor) if s])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, prop, title in [(axes[0], prop_before, "as simulated (no true change)"), (axes[1], prop_after, "after removing half of stimulated donors' NK cells")]:
    bottom = np.zeros(len(prop))
    for k, ct in enumerate(PBMC_TYPES):
        ax.bar(range(len(prop)), prop[ct].values, bottom=bottom, color=plt.get_cmap("tab10")(k), label=ct)
        bottom += prop[ct].values
    ax.set_xticks(range(len(prop))); ax.set_xticklabels([f"{d}\n{'stim' if s else 'ctrl'}" for d, s in zip(prop.index, is_stim)], fontsize=8)
    ax.set_title(f"SIMULATED {title}", fontsize=9)
axes[0].set_ylabel("proportion of cells"); axes[1].legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.show()

In [ ]:
# ── 8.2 🧪 A per-type test with donors as replicates (Welch t-test on proportions, 3 vs 3) ──
rows = {}
for ct in PBMC_TYPES:
    for name, prop in [("before", prop_before), ("after", prop_after)]:
        t = stats.ttest_ind(prop.loc[is_stim, ct], prop.loc[~is_stim, ct], equal_var=False)
        rows.setdefault(ct, {})[f"stim/ctrl ratio ({name})"] = prop.loc[is_stim, ct].mean() / prop.loc[~is_stim, ct].mean()
        rows[ct][f"p ({name})"] = t.pvalue
table = pd.DataFrame(rows).T
table["true change in absolute numbers"] = ["none"] * len(table)
table.loc["NK", "true change in absolute numbers"] = "−50 % (planted)"
print(table.round(3).to_string())

🔍 **What just happened?** Before the planted change no cell type differs significantly between conditions (all
p-values above 0.25) — correct, since the simulation has no composition effect. After removing half of the stimulated
donors' NK cells, the NK proportion halves (ratio 0.47, p = 0.003) and is detected easily even with three donors per
group. But look at the other rows: every other type's share went *up* by the same factor (about 7.5 %), and for the two
large, stable populations this apparent increase even reaches "significance" (CD4 T cells p = 0.03, monocytes
p = 0.016) — although their absolute numbers never changed. A biologist reading the table naïvely would report that
stimulation expands monocytes and CD4 T cells.

Two remedies are common. With an external measure of absolute cell numbers (e.g. cell counts per microlitre of blood
from a clinical blood count, or spike-in cells), compositional data can be converted back to absolute abundances.
Without one, compositional methods analyse *ratios* between types or relative to a reference type that is assumed
unchanged (scCODA is a Bayesian model of this kind), and neighbourhood-based methods (Milo) test abundance changes along
the k-NN graph without discrete clusters. In every case the replicates are **donors**: counting cells as replicates
(e.g. a χ² test on the pooled table) confuses sampling noise of cells with biological variation between people — the
pseudoreplication error once more.

✏️ **Your turn.** Replace the t-test on proportions by a t-test on log-ratios to CD4 T cells
(`np.log(prop[ct] / prop["CD4 T"])`). Which types are now "changed", and what assumption about CD4 T cells did you make?

### 📄 Research connection — Comparing conditions in single-cell studies

- **What the field does.** Multi-sample, multi-condition single-cell studies (patients vs controls, stimuli, perturbations) compare both
    expression and composition. For expression, aggregating to pseudobulk per sample and cell type and using bulk
    methods controls false discoveries (Crowell *et al.* 2020; Squair *et al.* 2021), whereas cell-level tests confuse
    cells with replicates. For composition, Bayesian compositional models (scCODA) and graph-neighbourhood tests (Milo)
    are used alongside simple per-type tests with samples as replicates.
- **What this notebook builds.** Pseudobulk NB GLMs with empirical-Bayes dispersions (Module 26) applied to all annotated cell types and
    graded against the true per-type effects, compared with cell-level Wilcoxon tests; a composition analysis with a
    planted change that exposes the compositional trap.
- **Where the frontier is.** Differential expression and abundance along continuous states rather than discrete clusters, designs
    with hundreds of donors (population-scale single-cell eQTL studies), causal readouts from pooled genetic screens
    (Perturb-seq, [Module 41](41_perturbation_causal_inference.ipynb)) and the integration of batches and studies
    ([Module 31](31_batch_integration.ipynb)) are the current directions.
- **Read next.** Squair *et al.* 2021 (*Nature Communications*), *Confronting false discoveries in single-cell differential expression*; Kang *et al.* 2018 (*Nature Biotechnology*), *Multiplexed droplet single-cell RNA-sequencing using natural genetic variation*.

---
## 9 · 🧭 From toy to real data

> 📍 **Where we are.** Every step above ran on 4,000 simulated cells with an answer key. Here is what changes with real
> data — no downloads, just a map for your first real analysis.

**Scale.** Real data sets have 20,000–30,000 genes and from thousands to millions of cells; the count matrix is stored
**sparse** (only non-zeros), and dense operations like our full residual matrix are replaced by sparse or chunked
code. Scanpy's AnnData objects and Seurat objects keep counts, normalised data, embeddings and annotations together.

**Public data to practise on.** The 10x Genomics PBMC data sets (e.g. "PBMC 3k" from
[10x Genomics datasets](https://www.10xgenomics.com/datasets)) are the "hello world" of single-cell analysis; the Kang
*et al.* (2018) interferon data are available through [GEO](https://www.ncbi.nlm.nih.gov/geo/) and many tutorials;
[CELLxGENE](https://cellxgene.cziscience.com/) and its programmatic
[Census](https://chanzuckerberg.github.io/cellxgene-census/) give access to tens of millions of standardised, annotated
cells; the [Human Cell Atlas Data Portal](https://data.humancellatlas.org/) hosts atlas projects.

**Production tools.** [Scanpy](https://scanpy.readthedocs.io/) (Python) and [Seurat](https://satijalab.org/seurat/) (R)
implement this whole notebook: `sc.pp.calculate_qc_metrics` (2.1), `sc.pp.scrublet` (2.3), `sc.pp.normalize_total` +
`sc.pp.log1p` (3.1), `sc.experimental.pp.normalize_pearson_residuals` (3.1), `sc.pp.highly_variable_genes` (3.2),
`sc.pp.pca` / `sc.pp.neighbors` / `sc.tl.tsne` or `sc.tl.umap` (4), `sc.tl.leiden` (5), `sc.tl.rank_genes_groups` (6),
and decoupler or muscat for pseudobulk (7). The [single-cell best practices](https://www.sc-best-practices.org/) online
book (Heumos *et al.* 2023) walks through all steps on real data.

**What goes wrong first.**

1. **Ambient RNA.** Free RNA from lysed cells is captured in every droplet, so marker genes of abundant types (e.g.
   haemoglobin, immunoglobulins) appear at low levels in all cells; tools such as SoupX and CellBender estimate and
   remove it. Empty droplets must be distinguished from small cells first.
2. **Batch effects.** Samples processed on different days or with different chemistry cluster by batch rather than by
   biology — the subject of [Module 31](31_batch_integration.ipynb). Always colour embeddings by sample and batch.
3. **Continuous biology and confounders.** Real cells form gradients (differentiation, activation — [Module 32](32_cell_dynamics_velocity.ipynb)),
   cell-cycle and stress signatures can dominate clusters, and doublets and damaged cells are less cleanly separable
   than in our simulation. Annotation then requires judgement, references and iteration rather than a single argmax.

**What simulation cannot teach.** Real data have no answer key; instead of 🧪 checks you rely on known biology
(canonical markers), consistency across samples and methods, and orthogonal validation (flow cytometry, imaging,
CITE-seq protein measurements — [Module 34](34_multimodal_integration.ipynb)).

## 📌 Key takeaways

- **Droplet scRNA-seq** labels each cell's mRNAs with a cell barcode and each molecule with a UMI, giving a sparse
  cells × genes matrix of molecule counts; its zeros are mostly what NB counting and cell-type heterogeneity predict,
  not a separate "dropout" process.
- **Quality control** flags damaged cells with robust MAD thresholds on counts, genes and % mitochondrial UMIs, and
  doublets with a Scrublet-style k-NN score against artificial doublets; homotypic doublets remain invisible.
- **Normalisation and HVGs:** library-size + log is a rough variance stabiliser; analytic Pearson residuals are the
  standardised residuals of an offset-only NB GLM; mean-aware HVG criteria recover far more marker genes than raw
  log-variance.
- **PCA + k-NN graph** is the backbone of the pipeline; **t-SNE/UMAP are for looking** — they keep neighbourhoods, not
  distances between clusters or cluster sizes.
- **Graph clustering** (modularity, Louvain/Leiden) finds clusters of any size, including rare types that k-means with a
  fixed K merges; the **resolution** sets granularity, and **a cluster is a hypothesis** — a type, a state, a cell-cycle
  phase or an artefact.
- **Annotation** combines rank-sum marker tests (useful for ranking, invalid as tests of cluster reality — double
  dipping), dot plots and marker scores; our marker-score annotation was > 99 % accurate on this clean simulation.
- **Condition comparisons** need donors as replicates: pseudobulk NB GLMs controlled false discoveries in every cell
  type, cell-level tests produced hundreds; **composition** is compositional — a loss of one type inflates the
  proportions of all others.

---
## ✅ Checkpoint

1. Why does a high mitochondrial percentage flag damaged cells?
   <details><summary>Answer</summary>When the cell membrane is damaged, cytoplasmic mRNAs leak out of the cell before capture, while mitochondrial transcripts are protected inside the mitochondria and stay. The remaining UMIs are therefore enriched for mitochondrial genes, and the total UMI count is low.</details>
2. Why do we take log(1 + x) of normalised counts rather than using them directly?
   <details><summary>Answer</summary>Counts have a variance that grows with the mean (Var ≈ μ + αμ²), so without a transformation the most highly expressed genes dominate distances and PCA. The log approximately stabilises the variance (Var(log y) ≈ 1/μ + α); the +1 pseudo-count keeps zeros finite (log 0 is −∞).</details>
3. What is an analytic Pearson residual, and what does a residual variance of 1 mean?
   <details><summary>Answer</summary>r = (y − μ̂)/√(μ̂ + μ̂²/θ), the standardised residual of an NB model in which every cell expresses the gene at the same proportion of its UMIs (μ̂ = n_c · p̂_g, library size as offset). Variance ≈ 1 means the gene varies across cells no more than counting noise predicts — no biological signal; HVGs have variance well above 1.</details>
4. Why should distances between clusters on a t-SNE map not be interpreted?
   <details><summary>Answer</summary>t-SNE only tries to keep each point's near neighbours; its heavy-tailed kernel pushes clusters apart by amounts that do not reflect their true separation, and it changes cluster sizes. In our 🧪 check the between-type distances correlated only loosely with those in PC space, while local neighbourhoods were well preserved.</details>
5. A doublet detector simulates artificial doublets from random pairs of droplets. Why is a doublet of two CD4 T cells much harder to detect than a CD4 T + monocyte doublet?
   <details><summary>Answer</summary>The sum of two CD4 T cells has a CD4-T-like profile — after normalisation it looks like one cell with more UMIs — so it has no hybrid signature placing it among the heterotypic artificial doublets. Only its depth (more UMIs, fewer zeros) distinguishes it; in our simulation that depth leak still flagged 61 % of homotypic artificial doublets, but on real data, where depth varies for many reasons, homotypic doublets are largely invisible to profile-based tools.</details>
6. A colleague compares stimulated vs control monocytes with a Wilcoxon test on 800 vs 750 cells from 3 + 3 donors and reports 600 DE genes at FDR 5 %. What is the unit of replication, and what would you do?
   <details><summary>Answer</summary>The donor (3 + 3). Cells of one donor are not independent; donor-to-donor differences masquerade as treatment effects. Aggregate counts per donor (pseudobulk) and fit an NB GLM (or a mixed model with a donor random effect); expect far fewer, but more trustworthy, genes.</details>
7. What does increasing the Louvain resolution γ do, and how would you choose it?
   <details><summary>Answer</summary>It increases the penalty γ k_i k_j/2m for putting cells in the same community, so the optimum has more, smaller communities. Modularity itself does not identify a correct γ; choose it by the biological granularity needed, check marker coherence and stability across subsamples or seeds, and treat clusters as hypotheses.</details>
8. In `rank_sum_test`, why is the tie term so important for single-cell data?
   <details><summary>Answer</summary>Most cells share the value 0 for most genes, so ranks contain huge ties. Without the correction Σ(t³ − t)/(n(n − 1)), the variance of U is overestimated (for the many tied zeros), the z-scores are too small and the p-values wrong; with it our function matches SciPy exactly.</details>

## 🧠 Questions & discussion

1. Our marker-score annotation reached > 99 % accuracy. List three properties of the simulation that made this easy, and for each one say how real PBMC data differ.
2. Clusters split stimulated from control monocytes. Should a cell atlas call them two cell types, two states of one type, or neither? What evidence would you want?
3. QC thresholds based on MADs adapt to each data set. Can you think of a tissue or disease in which an adaptive threshold would remove precisely the biologically interesting cells?
4. Single-cell atlases such as the Human Cell Atlas sample donors of different ancestries, ages and sexes. What could go wrong if an annotation reference is built mostly from one population, and who might be harmed?
5. Pseudobulk analysis needs enough cells of a type in every donor. How would you analyse a rare type with 5–20 cells per donor?
6. t-SNE and UMAP plots appear in nearly every single-cell paper. Which statements in a paper's text would you accept on the basis of such a plot alone, and which would you ask to see backed by an analysis in PC or graph space?

## 🚀 Bonus — going further

- **Leiden vs Louvain.** Implement the refinement idea of Leiden in a simplified form — after Louvain, split each community into its connected components in the induced subgraph — and check whether any community was disconnected at resolution 2.0.
- **Cell-cycle scoring.** Score each cell by the mean z-scored expression of the toolkit's `CYCLING_GENES`, compare with `cells_obs.is_cycling` (precision/recall), and check which Louvain clusters at high resolution are cycling cells.
- **Sub-clustering T cells.** Re-run HVG selection, PCA and Louvain on the CD4 and CD8 T cells only. Does the separation improve when the other types no longer dominate the variance?
- **A GMM with K chosen by BIC** ([Module 13](13_mixture_models_em.ipynb)): fit `GaussianMixture` for K = 3…12 on the 15 PCs and compare the BIC choice with the true number of types and with Louvain.
- **Try the real thing.** Install Scanpy, load the 10x PBMC 3k data set (its tutorial does), and run the same pipeline; compare your clusters and markers with the published annotation.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 30 — Single-cell RNA-seq: from counts to cell types       Date:
Biological question: which immune cell types are in blood, how does each respond to interferon, did the mix change?
Data (SIMULATED): simulate_scrna (4,000 cells × 1,500 genes, 6 donors, 3 ctrl + 3 IFN-stimulated; truth: true_type,
                  is_doublet, is_damaged, is_cycling, marker_of, true_logfc)
Method: MAD QC; Scrublet-style doublet scores; log-normalisation and analytic Pearson residuals; HVGs; PCA; k-NN graph;
        t-SNE; Louvain (vs k-means, GMM); Wilcoxon markers; marker-score annotation; pseudobulk NB GLM; composition tests
Key equation: r = (y − μ̂)/√(μ̂ + μ̂²/θ), μ̂ = n_c p̂_g;   Q = (1/2m) Σ_ij [A_ij − γ k_i k_j/2m] δ(c_i, c_j)
Key parameters: MAD multipliers; doublet threshold; number of HVGs and PCs; k of the k-NN graph; resolution γ
Ground-truth check result: damaged P/R …; doublet P/R …; ARI Louvain/k-means …; annotation accuracy …; DE FDR …
What changed when I modified …: (e.g. perplexity, resolution, theta, n_clusters)
One failure mode: (e.g. k-means merging a rare type; cell-level DE false discoveries; compositional artefacts)
Connection to research (paper/tool): Scanpy/Seurat; Lause et al. 2021; Traag et al. 2019; Squair et al. 2021
```

## 📚 References

- Regev A. *et al.* (2017) The Human Cell Atlas. *eLife* 6:e27041.
- Zheng G. X. Y. *et al.* (2017) Massively parallel digital transcriptional profiling of single cells. *Nature Communications* 8:14049.
- Kang H. M. *et al.* (2018) Multiplexed droplet single-cell RNA-sequencing using natural genetic variation. *Nature Biotechnology* 36:89–94.
- Luecken M. D., Theis F. J. (2019) Current best practices in single-cell RNA-seq analysis: a tutorial. *Molecular Systems Biology* 15:e8746.
- Heumos L. *et al.* (2023) Best practices for single-cell analysis across modalities. *Nature Reviews Genetics* 24:550–572.
- Wolf F. A., Angerer P., Theis F. J. (2018) SCANPY: large-scale single-cell gene expression data analysis. *Genome Biology* 19:15.
- Hao Y. *et al.* (2021) Integrated analysis of multimodal single-cell data. *Cell* 184:3573–3587.
- Wolock S. L., Lopez R., Klein A. M. (2019) Scrublet: computational identification of cell doublets in single-cell transcriptomic data. *Cell Systems* 8:281–291.
- Svensson V. (2020) Droplet scRNA-seq is not zero-inflated. *Nature Biotechnology* 38:147–150.
- Hafemeister C., Satija R. (2019) Normalization and variance stabilization of single-cell RNA-seq data using regularized negative binomial regression. *Genome Biology* 20:296.
- Lause J., Berens P., Kobak D. (2021) Analytic Pearson residuals for normalization of single-cell RNA-seq UMI data. *Genome Biology* 22:258.
- Ahlmann-Eltze C., Huber W. (2023) Comparison of transformations for single-cell RNA-seq data. *Nature Methods* 20:665–672.
- Blondel V. D., Guillaume J.-L., Lambiotte R., Lefebvre E. (2008) Fast unfolding of communities in large networks. *Journal of Statistical Mechanics* P10008.
- Traag V. A., Waltman L., van Eck N. J. (2019) From Louvain to Leiden: guaranteeing well-connected communities. *Scientific Reports* 9:5233.
- Levine J. H. *et al.* (2015) Data-driven phenotypic dissection of AML reveals progenitor-like cells that correlate with prognosis. *Cell* 162:184–197.
- van der Maaten L., Hinton G. (2008) Visualizing data using t-SNE. *JMLR* 9:2579–2605.
- Kobak D., Berens P. (2019) The art of using t-SNE for single-cell transcriptomics. *Nature Communications* 10:5416.
- Chari T., Pachter L. (2023) The specious art of single-cell genomics. *PLoS Computational Biology* 19:e1011288.
- Squair J. W. *et al.* (2021) Confronting false discoveries in single-cell differential expression. *Nature Communications* 12:5692.
- Crowell H. L. *et al.* (2020) muscat detects subpopulation-specific state transitions from multi-sample multi-condition single-cell transcriptomics data. *Nature Communications* 11:6077.

## ➡️ Next

Our simulation had a single batch. Real studies combine samples processed on different days, with different
chemistries, in different labs — and those differences can dominate the biology.
[Module 31 · Batch effects, data integration and reference mapping](31_batch_integration.ipynb) shows how to detect,
correct and avoid over-correcting them, and how to annotate new data by mapping onto a reference atlas.